# Landsat vegetation and water indices over the Ogallala aquifer: every scene, 500 m cell means, one CF file per path/row

Landsat 4, 5, 7, 8 and 9 (Collection 2 Level-2 surface reflectance, Tier 1), 1982 → now. The 30 m pixels are
**downloaded to Colab's scratch disk and the indices are computed here**, in half precision (FP16), using
eemont's index formulas. Earth Engine only hands out its stored integers, so almost none of the project's
Earth Engine compute quota is used.

| | |
|---|---|
| Output | `MyDrive/MSUGWB/LANDSAT_C2L2_indices_500m/LANDSAT_C2L2_indices_500m_p<path>r<row>_Ogallala.nc` |
| One record | one Landsat scene (nothing is composited in time) |
| Grid | the USGS High Plains 500 m grid (NAD83 / Conus Albers), the grid of `USGS_HPA_water_level_change_500m.nc` |
| Cells kept | cells whose centre lies inside the aquifer polygon |
| Variables | 18 indices, `n_clear`, `n_valid`, and per scene: satellite, cloud cover, sun elevation, scene ids |

**Indices** (set in Cell 2): NDVI, EVI, EVI2, SAVI, MSAVI, OSAVI, GNDVI, NIRv, kNDVI · NDMI, NMDI, MSI, GVMI ·
NDWI, MNDWI, AWEInsh, AWEIsh, WI2015. LSWI and NDII are the same formula as NDMI and are not stored twice.

**How to run**

1. Runtime → Change runtime type → a **GPU** (T4 is enough). It works on a CPU runtime too, only slower.
2. *Run all* with `TEST_MODE = True`: 2 path/rows × 6 scenes from three summers into `.../_test/`, every
   check switched on, and a projection of runtime, download volume and Drive size for the whole record.
3. Set `TEST_MODE = False` and *Run all*. If the runtime dies or you stop it, run all again: the run
   continues from what is on Drive (see *Stopping and resuming*). `ONLY_PATHROWS` lets several Colab
   sessions work on different path/rows; never run two sessions on the same path/row.

**How it works** (download to scratch → check → index maths → checkpoint on Drive → delete → repeat)

* The scene list comes from Earth Engine's catalogue and is cached on Drive. Scenes are grouped by WRS-2
  path/row; each path/row gets one file and one fixed window of the 500 m grid.
* A scene is fetched in blocks of its **own 30 m grid**, up to 32 at a time, as the stored 16-bit
  integers: the two quality bands and the six reflective bands in one request per block, because Earth
  Engine takes several seconds per request whatever its size. Earth Engine refuses a request above
  48 MiB, counting 3 bytes per band value, so a block holds at most 1448 × 1448 pixels; each scene is cut
  into the fewest such rectangles (about 28 to 30). Every block is checked when it arrives and again
  when it is read back from scratch. A scene that cannot be fetched stops the run instead of leaving a hole.
* A pixel is used unless it is fill, dilated cloud, cirrus, cloud, cloud shadow or snow (`QA_PIXEL`),
  saturated (`QA_RADSAT`), or has a reflective band outside USGS's valid range 7273–43636. The last rule
  matters: a few dark pixels with zero or negative reflectance otherwise push a cell's mean NDVI to values
  like −7.
* Each usable pixel goes to the 500 m cell that holds its centre. The cell value is the plain mean of the
  index over those pixels; `n_clear` says how many were averaged and `n_valid` how many the scene has
  there (a full cell holds about 278), so a minimum can be chosen later.
* Records are read back from the file and compared with memory before the raw blocks are deleted.
  At least 20 GB of scratch stays free.

**Stopping and resuming**

* Every finished group of scenes (`GROUP_SCENES`, 16 by default) is saved to Drive at once, as a small
  file in `_checkpoints/p<path>r<row>/`. A runtime that dies loses at most the group it was working on.
* A new run trusts only Drive: the path/row's file, if it has one, followed by its checkpoints in order.
  The log says where it continues: `RESUME: Drive holds … records through <date>`.
* When the last scene of a path/row is done, its checkpoints are joined into the path/row's single
  `.nc`, and every record taken from a checkpoint is compared with it, bit for bit. The checkpoints are
  deleted by a later look, at least 30 minutes after the file was uploaded.
* **Each path/row has its own file and its own start in the 1980s.** When one path/row is finished the
  next one begins at its first scene. That is a new file for a new place, not a restart; the log line is
  `nothing on Drive for this path/row yet`.
* The stop button no longer waits for the downloads that are still queued. After a stop, give Drive a
  minute to finish uploading before you disconnect the runtime.
* If a file on Drive cannot be read, the run stops and changes nothing. It never starts a path/row over
  on its own.
* Files deleted from Drive normally go to its Trash, stay there for 30 days and still count against
  your storage. The checkpoints of every finished path/row end up there, so empty the Trash now and then
  during a long run.

**What eemont does here.** eemont itself can only compute on Google's servers, and that is the part that
costs quota. So this notebook takes from eemont what defines the result: the formula, name and reference
of every index (`eemont.indices()`), and each band's scale and offset (`getScaleParams()`,
`getOffsetParams()`). It then evaluates those formulas locally. Before any bulk work, a small window of
one scene per satellite is also computed by eemont on Earth Engine (`scaleAndOffset()`,
`spectralIndices()`) and compared pixel by pixel; the two must agree to rounding or the run stops.

**Two places where eemont's defaults are not used** (both are settings in Cell 2)

* `SAVI` is computed with L = 0.5 as published. eemont shares one `L` between EVI and SAVI and defaults to 1.
* `AWEInsh` uses the formula of Feyisa et al. (2014), `4(G − S1) − (0.25 N + 2.75 S2)`. eemont's
  catalogue has `+ 2.75 S2`.

**Precision.** Reflectance and index values of the pixels are 16-bit floats. Sums are kept in 64-bit, so
averaging adds nothing; the mean is then stored as a 16-bit float value inside a 32-bit variable, because
netCDF has no 16-bit float type. That is about 3 significant digits (steps of 0.0005 for a value near
0.5) and roughly halves the file size. `HALF_PRECISION = False` switches to 32-bit throughout.

**What is not done.** No compositing, no gap filling (Landsat 7 stripes after May 2003 stay empty), no
harmonisation between TM/ETM+ and OLI. EVI can take very large values where its denominator passes
through zero on bright surfaces; screen EVI outside −1…1 before use.

In [ ]:
# ═══════════════ CELL 1: install ═══════════════
# earthengine-api, numpy, cupy (on GPU runtimes), geopandas, shapely and pyproj are part of Colab.
# If Colab offers "RESTART RUNTIME" after this cell, restart, then Run all again.
!pip install -q "eemont==2025.7.1" netCDF4 rasterio 2>&1 | tail -2

In [ ]:
# ═══════════════ CELL 2: mount Drive, stage the shapefile, SETTINGS ═══════════════
import os, zipfile
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
TEST_MODE     = True       # True -> short rehearsal into .../_test/ ; False -> the whole record
TEST_PATHROWS = 2          # path/rows in the rehearsal (one mostly inside the aquifer, one on its edge)
TEST_SCENES   = 6          # scenes per path/row in the rehearsal, spread over the windows below
TEST_TEMPORAL = [("1990-06-01", "1990-09-01"), ("2005-06-01", "2005-09-01"), ("2022-06-01", "2022-09-01")]
TEST_WINDOW_CELLS = None   # None -> whole path/row; or e.g. 96 to rehearse on a 48 km square only

PROJECT_ID = "msugw-503806"                                    # Earth Engine cloud project
EE_URL     = "https://earthengine-highvolume.googleapis.com"   # endpoint meant for many parallel requests
SATELLITES = ["LT04", "LT05", "LE07", "LC08", "LC09"]          # Landsat 4, 5, 7, 8, 9 (Collection 2 Level-2)
TIER       = "T1"                                              # Tier 1 only (USGS's time-series quality)
DATE_START = None          # None -> first acquisition; or "YYYY-MM-DD"
DATE_END   = None          # None -> now
ONLY_PATHROWS = None       # None -> every path/row; or e.g. ["031034", "030034"] (re-do one, or split sessions)
SKIP_SCENES   = []         # scene ids to treat as permanently unavailable (see the stall message)

# ---- indices (formulas, names and references come from eemont's catalogue) ----
INDICES = ["NDVI", "EVI", "EVI2", "SAVI", "MSAVI", "OSAVI", "GNDVI", "NIRv", "kNDVI",     # greenness
           "NDMI", "NMDI", "MSI", "GVMI",                                                  # canopy water
           "NDWI", "MNDWI", "AWEInsh", "AWEIsh", "WI2015"]                                 # open water
INDEX_PARAMS     = dict(g=2.5, C1=6.0, C2=7.5, L=1.0)   # eemont's defaults (EVI, EVI2)
INDEX_PARAMS_FOR = {"SAVI": dict(L=0.5)}                # SAVI as published; eemont's shared default L=1 is EVI's
AWEINSH_AS_PUBLISHED = True   # True -> 4*(G-S1) - (0.25*N + 2.75*S2) as in Feyisa et al. 2014;
                              # False -> eemont's catalogue formula, which has "+ 2.75*S2"

# ---- which 30 m pixels enter a cell mean ----
QA_PIXEL_REJECT  = 0b111110   # QA_PIXEL bits 1-5: dilated cloud, cirrus, cloud, cloud shadow, snow
REJECT_SATURATED = True       # leave out pixels with any QA_RADSAT bit set
REQUIRE_VALID_RANGE = True    # leave out pixels with a reflective band outside USGS's valid range 7273..43636

# ---- precision ----
HALF_PRECISION = True      # True -> reflectance and index values are held in 16-bit floats (FP16)
USE_GPU        = "auto"    # "auto" -> CuPy on the GPU when the runtime has one, otherwise NumPy on the CPU

# ---- download and compute ----
ONE_REQUEST_PER_BLOCK = True   # True -> quality + reflectance bands in one request per block (fewer, larger
                               # requests: faster); False -> quality bands first, reflectance only where needed
BLOCK_PX           = 1448  # a download block holds at most BLOCK_PX x BLOCK_PX pixels. Earth Engine counts 3 bytes
                           # per band value and refuses a request above 48 MiB, which for 8 bands is 1448 x 1448.
                           # A larger value here is cut back to that limit by itself
N_DOWNLOAD_THREADS = 32    # blocks downloaded at a time (Earth Engine allows 40 concurrent requests)
GROUP_SCENES       = 16    # scenes downloaded to scratch, computed and saved to Drive as one checkpoint per cycle
                           # (a runtime that dies loses at most the group it was working on)
COMPUTE_THREADS    = 2     # scenes computed at a time on the CPU (1 is used on a GPU)
REQUEST_DEADLINE_S = 180   # give up on a single request after this long (it is then retried)

COMPRESS_LEVEL = 1         # zlib level of the CF .nc (lossless)
CHUNK_YX       = 256       # HDF5 chunk edge in y and x
WRITE_LATLON   = True      # also store 2-D lat/lon auxiliary coordinates

MIN_FREE_GB       = 20.0   # scratch space that must stay free at all times
DISK_MAX_WAIT_S   = 2 * 3600
KEEP_CHECKPOINTS_MIN = 30  # a finished path/row's checkpoints leave Drive only this long after its file was
                           # uploaded (Drive uploads in the background; a runtime may die before it is done)
DRIVE_READ_TRIES  = 4      # attempts at reading a file's state from Drive before the run stops. The run never
                           # starts a path/row over because a file on Drive could not be read
MAX_RETRIES       = 4      # extra attempts per block
STALL_MINUTES     = 20     # stop cleanly after this long with no progress at all
READBACK_CHECKS   = 3      # records per file re-read and compared with memory (all in TEST_MODE)
EEMONT_CHECK_PX   = 48     # edge of the window compared pixel by pixel with eemont on Earth Engine

MANIFEST_MAX_AGE_DAYS = 7  # refresh the cached scene list when older than this
REBUILD_MANIFEST      = False
# =======================================================================

In [ ]:
# ═══════════════ CELL 3: imports, logging, paths, Earth Engine sign-in, eemont definitions, engine ═══════════════
import re, gc, io, ast, sys, json, gzip, glob, time, math, shutil, hashlib, logging, threading, warnings
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
from numpy.lib import recfunctions as rfn
import netCDF4 as nc4
import geopandas as gpd
import pyproj
import shapely
import rasterio.features
from affine import Affine
import ee
import eemont                      # index catalogue, scale/offset tables, and the server-side reference

# force=True: Colab installs its own root log handler, so a plain basicConfig() is a no-op there.
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("landsat")
warnings.filterwarnings("ignore", category=UserWarning, module="ee_extra")
for _n in ("urllib3.connectionpool", "googleapiclient", "google_auth_httplib2", "rasterio", "pyogrio"):
    logging.getLogger(_n).setLevel(logging.ERROR)

# ---------- names and paths (Drive = durable, scratch = fast and disposable) ----------
ROI_LABEL   = "Ogallala"
PRODUCT_TAG = "LANDSAT_C2L2_indices_500m"
_SUFFIX     = "_test" if TEST_MODE else ""          # the rehearsal never touches real outputs
DRIVE_ROOT  = "/content/drive/MyDrive/MSUGWB"
DRIVE_OUT   = f"{DRIVE_ROOT}/{PRODUCT_TAG}" + ("/_test" if TEST_MODE else "")
SCRATCH     = "/content/landsat_scratch" + _SUFFIX
RAW_DIR     = f"{SCRATCH}/raw"                      # downloaded 30 m blocks, deleted once a scene is written
NC_DIR      = f"{SCRATCH}/nc"
DISK_PATH   = "/content"                            # same filesystem as scratch AND Drive's local upload cache

CKPT_FOLDER   = "_checkpoints"                      # on Drive: one small file per finished group of scenes
MANIFEST_NAME = "manifest_landsat_c2l2.json.gz"
LOG_FAILED    = "failed_scenes.jsonl"
LOG_SKIPPED   = "skipped_scenes.jsonl"
LOG_FILES     = "pathrow_summary.jsonl"
for _d in (DRIVE_OUT, RAW_DIR, NC_DIR):
    os.makedirs(_d, exist_ok=True)

# ---------- the common 500 m grid (USGS High Plains rasters, NAD83 / Conus Albers) ----------
# Origin and size are those of USGS's own water-level-change raster hp_wlcpd19t.tif, i.e. the grid of
# merged_datasets/USGS_HPA_native/USGS_HPA_water_level_change_500m.nc. Cells line up one to one.
GRID_CRS = "EPSG:5070"
GX0, GY0, CELL = -805708.3125, 2312757.8125, 500.0   # upper-left corner (m) and cell size (m)
GRID_NY, GRID_NX = 2659, 1569
WINDOW_PAD = 4                                       # cells added around a path/row's footprint

TIME_UNITS, TIME_CALENDAR = "seconds since 2000-01-01 00:00:00", "proleptic_gregorian"
EPOCH = datetime(2000, 1, 1)

SAT_NAME   = {"LT04": "Landsat 4 TM", "LT05": "Landsat 5 TM", "LE07": "Landsat 7 ETM+",
              "LC08": "Landsat 8 OLI", "LC09": "Landsat 9 OLI-2"}
SAT_NUMBER = {"LT04": 4, "LT05": 5, "LE07": 7, "LC08": 8, "LC09": 9}
SAT_DOI    = {"LT04": "10.5066/P9IAXOVV", "LT05": "10.5066/P9IAXOVV", "LE07": "10.5066/P9C7I13B",
              "LC08": "10.5066/P9OGBGM6", "LC09": "10.5066/P9OGBGM6"}
# Band letters of the index catalogue -> Landsat Collection 2 Level-2 band names. Checked against
# eemont's own server-side result in eemont_cross_check() (Cell 7): a wrong letter fails that check.
BAND_LETTERS = {"OLI": dict(B="SR_B2", G="SR_B3", R="SR_B4", N="SR_B5", S1="SR_B6", S2="SR_B7"),
                "TM":  dict(B="SR_B1", G="SR_B2", R="SR_B3", N="SR_B4", S1="SR_B5", S2="SR_B7")}
SENSOR = {"LT04": "TM", "LT05": "TM", "LE07": "TM", "LC08": "OLI", "LC09": "OLI"}
LETTERS = ("B", "G", "R", "N", "S1", "S2")
SR_VALID = (7273, 43636)                             # USGS valid range of the reflective bands (DN)
QA_BANDS = ["QA_PIXEL", "QA_RADSAT"]


class PipelineStop(RuntimeError):
    """Clean stop: everything written so far is safe, re-running resumes."""


class DataIntegrityError(RuntimeError):
    """Something that must never reach the output was detected."""


class SceneUnavailable(Exception):
    """Earth Engine says the scene does not exist. Nothing else may be mistaken for this: a scene
    reported unavailable is left out of the record for good."""


class RunStopping(Exception):
    """Raised inside a download thread when the run is being stopped."""


class RequestTooLarge(Exception):
    """Earth Engine refused a pixel request for its size. Trying again cannot help; the rectangle
    is fetched as two halves instead."""


def utcnow():
    return datetime.now(timezone.utc).replace(tzinfo=None)   # naive UTC everywhere


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"


def parse_iso(s):
    s = s.rstrip("Z")
    return datetime.strptime(s, "%Y-%m-%dT%H:%M:%S.%f" if "." in s else "%Y-%m-%dT%H:%M:%S")


# ---------- Earth Engine ----------
def ee_login():
    """Stored credentials first (an unattended restart needs no clicks); otherwise the sign-in flow."""
    try:
        ee.Initialize(project=PROJECT_ID, opt_url=EE_URL)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=PROJECT_ID, opt_url=EE_URL)
    ee.data.setDeadline(REQUEST_DEADLINE_S * 1000)
    if ee.Number(1).add(1).getInfo() != 2:
        raise PipelineStop("Earth Engine did not answer a trivial request.")
    log.info("Earth Engine ready (project %s, high-volume endpoint)", PROJECT_ID)


def collection_id(sat):
    return f"LANDSAT/{sat}/C02/{TIER}_L2"


# ---------- index definitions, taken from eemont ----------
CATALOGUE = eemont.indices()                # eemont's bundled Awesome Spectral Indices catalogue
_KERNEL_RE = re.compile(r"^k(S1|S2|[BGRN])(S1|S2|[BGRN])$")
_AST_OK = (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant, ast.Name, ast.Load,
           ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Pow, ast.USub, ast.UAdd)


def _compile_formula(formula):
    """Catalogue formulas are plain arithmetic. Anything else is refused before it is evaluated."""
    tree = ast.parse(formula, mode="eval")
    names = set()
    for node in ast.walk(tree):
        if not isinstance(node, _AST_OK):
            raise ValueError(f"formula {formula!r} contains {type(node).__name__}")
        if isinstance(node, ast.Constant) and not isinstance(node.value, (int, float)):
            raise ValueError(f"formula {formula!r} contains a non-numeric constant")
        if isinstance(node, ast.Name):
            names.add(node.id)
    return compile(tree, "<index formula>", "eval"), names


def build_index_defs():
    defs = {}
    for name in INDICES:
        if name not in CATALOGUE:
            raise PipelineStop(f"'{name}' is not in eemont's index catalogue (see eemont.listIndices()).")
        entry = CATALOGUE[name]
        formula, origin = str(entry["formula"]), f"eemont {eemont.__version__} catalogue"
        if name == "AWEInsh" and AWEINSH_AS_PUBLISHED:
            formula = "4.0 * (G - S1) - (0.25 * N + 2.75 * S2)"
            origin = ("Feyisa et al. 2014 as published; eemont's catalogue formula reads "
                      f"'{entry['formula']}' (opposite sign on the SWIR2 term)")
        code, names = _compile_formula(formula)
        params = dict(INDEX_PARAMS, **INDEX_PARAMS_FOR.get(name, {}))
        kernels, used = {}, {}
        for nm in sorted(names):
            if nm in LETTERS:
                continue
            m = _KERNEL_RE.match(nm)
            if m:
                kernels[nm] = (m.group(1), m.group(2))
            elif nm in params:
                used[nm] = float(params[nm])
            else:
                raise PipelineStop(f"{name}: '{nm}' in '{formula}' is neither a Landsat band letter nor a "
                                   "parameter in INDEX_PARAMS. Landsat cannot supply it.")
        letters = sorted({n for n in names if n in LETTERS} | {x for ab in kernels.values() for x in ab})
        defs[name] = dict(code=code, formula=formula, origin=origin, names=sorted(names), params=used,
                          kernels=kernels, letters=letters, long_name=str(entry["long_name"]),
                          reference=str(entry["reference"]), domain=str(entry["application_domain"]),
                          catalogue_formula=str(entry["formula"]))
    return defs


INDEX_DEFS = build_index_defs()


def eval_index(d, bands, xp):
    """One index from reflectance arrays keyed by band letter. Works on NumPy and CuPy arrays alike and
    keeps their precision. Kernel terms follow eemont's default: RBF with sigma = 0.5 * (a + b)."""
    ns = {}
    for nm in d["names"]:
        if nm in bands:
            ns[nm] = bands[nm]
        elif nm in d["params"]:
            ns[nm] = d["params"][nm]
        else:
            a, b = d["kernels"][nm]
            if a == b:
                ns[nm] = 1.0
            else:
                sigma = 0.5 * (bands[a] + bands[b])
                ns[nm] = xp.exp(-((bands[a] - bands[b]) ** 2) / (2.0 * sigma ** 2))
    # The formula was reduced to plain arithmetic on these names by _compile_formula(), so nothing else can
    # be reached from it. (An emptied __builtins__ here would break CuPy, which imports while it computes.)
    return eval(d["code"], {}, ns)


# ---------- compute engine: CuPy on a GPU, NumPy otherwise ----------
def pick_engine():
    if USE_GPU in ("auto", True):
        try:
            import cupy as cp
            if cp.cuda.runtime.getDeviceCount() > 0:
                _ = float(cp.arange(4, dtype=cp.float16).sum())      # forces the CUDA context
                name = cp.cuda.runtime.getDeviceProperties(0)["name"]
                return cp, f"GPU ({name.decode() if isinstance(name, bytes) else name}, CuPy {cp.__version__})"
        except Exception as e:
            if USE_GPU is True:
                raise PipelineStop(f"USE_GPU is True but no GPU could be used: {e}")
    return np, f"CPU (NumPy {np.__version__})"


XP, ENGINE = pick_engine()
CDTYPE = np.float16 if HALF_PRECISION else np.float32       # precision of reflectance and index values
log.info("Engine: %s | pixel maths in %s | %d indices: %s", ENGINE, np.dtype(CDTYPE).name, len(INDICES),
         " ".join(INDICES))

In [ ]:
# ═══════════════ CELL 4: disk floor, verified Drive copy, logs, stall watchdog ═══════════════
_LOG_LOCK   = threading.Lock()
_DRIVE_LOCK = threading.Lock()
DRIVE_SETTLE_S = 10                      # let the Drive FUSE mount settle before re-reading
TIMES = {}                               # seconds spent per stage (reported at the end of the run)
TOTALS = {}                              # counters (bytes downloaded, files, records) for the summary


class timed:
    """with timed("download"): ...  adds the elapsed seconds to TIMES["download"]."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        TIMES[self.name] = TIMES.get(self.name, 0.0) + time.time() - self.t0
        return False


def log_events(name, entries):
    """Append JSON lines to a log on Drive in one write + fsync (survives a dead runtime)."""
    if not entries:
        return
    stamp = iso(utcnow())
    text = "".join(json.dumps(dict(e, logged=stamp), default=str) + "\n" for e in entries)
    with _LOG_LOCK:
        with open(os.path.join(DRIVE_OUT, name), "a") as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())


def human(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


# ---------- 20 GB scratch floor ----------
def free_gb():
    return shutil.disk_usage(DISK_PATH).free / 1024 ** 3


def ensure_disk_free(context="", extra_gb=0.0):
    """Block until MIN_FREE_GB + extra_gb is free. Scratch only ever holds the file being built,
    so a shortfall normally means Drive's local upload cache is still draining: wait for it
    rather than fail. Raises PipelineStop if it never clears."""
    need = MIN_FREE_GB + max(0.0, extra_gb)
    t0, warned = time.time(), False
    while True:
        free = free_gb()
        if free >= need:
            if warned:
                log.info("Disk OK again: %.1f GB free (need %.1f)", free, need)
            return free
        if not warned:
            warned = True
            log.warning("LOW DISK %.1f < %.1f GB (%s) - waiting for Drive to upload", free, need, context)
        if time.time() - t0 > DISK_MAX_WAIT_S:
            raise PipelineStop(f"Free space stayed at {free:.1f} GB (< {need:.1f} GB) for "
                               f"{DISK_MAX_WAIT_S // 60} min ({context}). Nothing is lost - re-run to resume.")
        time.sleep(60)


# ---------- verified copies to / from Drive ----------
def md5sum(path, block=10 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    """Copy src to Drive under a temporary name, re-read it, compare size + MD5, then
    rename over dst. A runtime that dies mid-copy leaves the previous good dst intact."""
    if os.path.abspath(src) == os.path.abspath(dst):
        raise ValueError(f"robust_drive_copy: source and destination are the same file ({src})")
    with _DRIVE_LOCK, timed("Drive copy"):
        size = os.path.getsize(src)
        ensure_disk_free("Drive copy", extra_gb=size / 1024 ** 3)   # FUSE caches the upload locally
        want = md5sum(src)
        tmp = dst + ".tmp"
        os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
        last = None
        for attempt in range(1, max_retries + 1):
            try:
                with open(src, "rb") as fs, open(tmp, "wb") as fd:
                    shutil.copyfileobj(fs, fd, length=10 * 1024 * 1024)
                    fd.flush()
                    os.fsync(fd.fileno())
                ok = False
                for _ in range(4):           # a just-closed FUSE file can return stale bytes
                    time.sleep(DRIVE_SETTLE_S)
                    try:
                        if os.path.getsize(tmp) == size and md5sum(tmp) == want:
                            ok = True
                            break
                    except OSError as e:
                        last = e
                if not ok:
                    raise IOError("size or MD5 mismatch after settle")
                os.replace(tmp, dst)
                log.info("Drive copy verified (%s): %s", human(size), os.path.basename(dst))
                return True
            except Exception as e:
                last = e
                log.warning("Drive copy attempt %d/%d failed: %s", attempt, max_retries, e)
                try:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                except OSError:
                    pass
                if attempt < max_retries:
                    time.sleep(10 * attempt)
        raise RuntimeError(f"Drive copy of {os.path.basename(src)} failed after {max_retries} attempts: {last}")


def restore_from_drive(src, dst):
    """Bring a Drive file back to scratch (size-checked, atomic)."""
    size = os.path.getsize(src)
    ensure_disk_free("restore from Drive", extra_gb=size / 1024 ** 3)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + ".part"
    shutil.copyfile(src, tmp)
    if os.path.getsize(tmp) != size:
        os.remove(tmp)
        raise IOError(f"restore of {os.path.basename(src)} is incomplete")
    os.replace(tmp, dst)
    log.info("Restored %s from Drive (%s)", os.path.basename(src), human(size))


# ---------- stop after a long stretch with no progress at all ----------
class StallWatchdog:
    def __init__(self, limit_min):
        self.limit = limit_min * 60.0
        self.since = None

    def progress(self):
        self.since = None

    def failure(self):
        """Record a cycle without progress; True once the stall has lasted too long."""
        now = time.time()
        if self.since is None:
            self.since = now
        return (now - self.since) > self.limit


STALL = StallWatchdog(STALL_MINUTES)

In [ ]:
# ═══════════════ CELL 5: aquifer mask on the 500 m grid, scene list, path/row windows, block plan ═══════════════
_TLS = threading.local()


def transformer(crs):
    """Scene CRS (UTM) -> grid CRS. One pyproj object per thread and CRS."""
    cache = getattr(_TLS, "tf", None)
    if cache is None:
        cache = _TLS.tf = {}
    if crs not in cache:
        cache[crs] = pyproj.Transformer.from_crs(crs, GRID_CRS, always_xy=True)
    return cache[crs]


def load_roi(shp_path):
    """Aquifer polygon -> boolean mask of the 500 m grid (cell centre inside the polygon), plus a
    rough lon/lat outline that is used only as the spatial filter of the catalogue listing."""
    gdf = gpd.read_file(shp_path)
    if gdf.crs is None:
        raise ValueError(f"{shp_path} has no CRS")

    def union(g):
        u = g.union_all() if hasattr(g, "union_all") else g.unary_union
        return u if u.is_valid else u.buffer(0)

    poly = union(gdf.to_crs(GRID_CRS).geometry)
    mask = rasterio.features.geometry_mask([poly], out_shape=(GRID_NY, GRID_NX), invert=True, all_touched=False,
                                           transform=Affine(CELL, 0.0, GX0, 0.0, -CELL, GY0))
    rough = union(gdf.to_crs(4326).geometry).buffer(0.15).simplify(0.10, preserve_topology=True)
    if rough.geom_type != "Polygon":
        rough = rough.convex_hull
    return mask, json.dumps(shapely.geometry.mapping(shapely.geometry.Polygon(rough.exterior)))


ROI_MASK, ROI_REGION = load_roi(SHP_PATH)
if not ROI_MASK.any():
    raise PipelineStop("The aquifer polygon does not touch the 500 m grid - check SHP_PATH and the grid constants.")
log.info("Aquifer: %d cells of 500 m (%.0f km2) on the %d x %d grid", int(ROI_MASK.sum()),
         ROI_MASK.sum() * CELL * CELL / 1e6, GRID_NY, GRID_NX)


def with_retries(fn, what, tries=None, stop=None):
    """Run an Earth Engine request; retry with growing pauses. A missing asset is not retried.
    `stop` (a threading.Event) ends the waiting at once when the run is being stopped."""
    n = 1 + (MAX_RETRIES if tries is None else tries)
    last = None
    for attempt in range(n):
        if stop is not None and stop.is_set():
            raise RunStopping(what)
        try:
            return fn()
        except Exception as e:
            last = e
            msg = str(e)
            low = msg.lower()
            if "request size" in low and "must be less than" in low:
                raise RequestTooLarge(msg[:200])
            if "asset" in low and ("not found" in low or "does not exist" in low):
                raise SceneUnavailable(f"{what}: {msg[:200]}")
            if attempt + 1 < n:
                quota = "quota" in msg.lower() or "429" in msg or "too many" in msg.lower()
                pause = min((20 if quota else 2) * 2 ** attempt, 300)
                if stop is None:
                    time.sleep(pause)
                elif stop.wait(pause):
                    raise RunStopping(what)
    raise IOError(f"{what}: {type(last).__name__}: {str(last)[:300]}")


# ---------- scene list (cached on Drive) ----------
def _scene_record(sat, im):
    """One catalogue entry -> what the pipeline needs. Returns (record, None) or (None, reason)."""
    p = im.get("properties", {})
    bands = {b["id"]: b for b in im.get("bands", [])}
    need = QA_BANDS + sorted(set(BAND_LETTERS[SENSOR[sat]].values()))
    if any(k not in bands for k in need):
        return None, f"bands missing: {[k for k in need if k not in bands]}"
    g = bands["QA_PIXEL"]["grid"]
    if any(bands[k]["grid"] != g for k in need):
        return None, "bands are not on one common grid"
    at = g["affineTransform"]
    if at.get("shearX", 0) or at.get("shearY", 0) or "WRS_PATH" not in p or "WRS_ROW" not in p:
        return None, "rotated grid or no WRS path/row"
    return dict(id=im["id"].split("/")[-1], name=im["name"], sat=sat, t=im["startTime"],
                path=int(p["WRS_PATH"]), row=int(p["WRS_ROW"]), cloud=float(p.get("CLOUD_COVER", -1)),
                sun_elev=float(p.get("SUN_ELEVATION", float("nan"))), product=str(p.get("LANDSAT_PRODUCT_ID", "")),
                updated=str(im.get("updateTime", "")), crs=g["crsCode"], tx=float(at["translateX"]),
                ty=float(at["translateY"]), sx=float(at["scaleX"]), sy=float(at["scaleY"]),
                w=int(g["dimensions"]["width"]), h=int(g["dimensions"]["height"])), None


def _list_collection(sat, t0, t1):
    parent = f"projects/earthengine-public/assets/{collection_id(sat)}"
    recs, odd, token = [], [], None
    while True:
        params = dict(parent=parent, region=ROI_REGION, pageSize=1000, view="FULL")
        if t0:
            params["startTime"] = f"{t0}T00:00:00Z"
        if t1:
            params["endTime"] = f"{t1}T00:00:00Z"
        if token:
            params["pageToken"] = token
        page = with_retries(lambda: ee.data.listImages(params), f"listing {sat}")
        for im in page.get("images", []):
            rec, why = _scene_record(sat, im)
            if rec is None:
                odd.append(dict(scene=im.get("id", "?"), reason=why))
            else:
                recs.append(rec)
        token = page.get("nextPageToken")
        if not token:
            return recs, odd


def get_manifest():
    """Every scene of the chosen satellites that touches the aquifer's outline, with its grid."""
    windows = TEST_TEMPORAL if TEST_MODE else [(DATE_START, DATE_END)]
    sig = json.dumps(dict(sats=SATELLITES, tier=TIER, windows=windows), sort_keys=True)
    path = os.path.join(DRIVE_OUT, MANIFEST_NAME)
    if os.path.exists(path) and not REBUILD_MANIFEST:
        try:
            with gzip.open(path, "rt") as f:
                m = json.load(f)
            age = (utcnow() - parse_iso(m["created"])).total_seconds() / 86400
            if m.get("signature") == sig and age <= MANIFEST_MAX_AGE_DAYS:
                log.info("Scene list from Drive: %d scenes, %.1f days old", len(m["scenes"]), age)
                return m["scenes"]
        except Exception as e:
            log.warning("Cached scene list unreadable (%s) - listing again", e)
    t0 = time.time()
    jobs = [(s, a, b) for s in SATELLITES for a, b in windows]
    recs, odd = [], []
    with ThreadPoolExecutor(min(8, len(jobs))) as ex:
        for r, o in ex.map(lambda j: _list_collection(*j), jobs):
            recs += r
            odd += o
    seen, uniq = set(), []
    for r in sorted(recs, key=lambda r: (r["t"], r["id"])):
        if r["id"] not in seen:
            seen.add(r["id"])
            uniq.append(r)
    log_events(LOG_SKIPPED, [dict(o, stage="scene list") for o in odd])
    local = os.path.join(NC_DIR, MANIFEST_NAME)
    with gzip.open(local, "wt") as f:
        json.dump(dict(created=iso(utcnow()), signature=sig, scenes=uniq), f)
    robust_drive_copy(local, path)
    os.remove(local)
    log.info("Scene list: %d scenes (%s), %d unusable entries, %.0f s", len(uniq),
             ", ".join(f"{s} {sum(1 for r in uniq if r['sat'] == s)}" for s in SATELLITES), len(odd), time.time() - t0)
    return uniq


# ---------- where a scene (or one block of it) falls on the 500 m grid ----------
def cell_range(r, row0=0, col0=0, h=None, w=None, margin=1):
    """Rows/cols of the grid touched by a pixel rectangle of scene r (corners and edge midpoints)."""
    h, w = r["h"] if h is None else h, r["w"] if w is None else w
    xs = r["tx"] + (col0 + np.array([0, w / 2, w, 0, w, 0, w / 2, w])) * r["sx"]
    ys = r["ty"] + (row0 + np.array([0, 0, 0, h / 2, h / 2, h, h, h])) * r["sy"]
    ax, ay = transformer(r["crs"]).transform(xs, ys)
    return (int(np.floor((GY0 - ay.max()) / CELL)) - margin, int(np.ceil((GY0 - ay.min()) / CELL)) + margin,
            int(np.floor((ax.min() - GX0) / CELL)) - margin, int(np.ceil((ax.max() - GX0) / CELL)) + margin)


def _clip(win):
    return max(win[0], 0), min(win[1], GRID_NY), max(win[2], 0), min(win[3], GRID_NX)


def build_pathrows(recs):
    """Group the scenes by WRS-2 path/row and give each path/row one fixed window of the grid: the
    union of its scenes' footprints, padded, then cropped to the aquifer cells it holds."""
    groups = {}
    for r in recs:
        groups.setdefault((r["path"], r["row"]), []).append(r)
    out = []
    for (p, w_), scenes in groups.items():
        boxes = np.array([cell_range(r) for r in scenes])
        r0, r1, c0, c1 = _clip((boxes[:, 0].min() - WINDOW_PAD, boxes[:, 1].max() + WINDOW_PAD,
                                boxes[:, 2].min() - WINDOW_PAD, boxes[:, 3].max() + WINDOW_PAD))
        if r0 >= r1 or c0 >= c1 or not ROI_MASK[r0:r1, c0:c1].any():
            continue
        sub = ROI_MASK[r0:r1, c0:c1]
        rows, cols = np.flatnonzero(sub.any(axis=1)), np.flatnonzero(sub.any(axis=0))
        win = (r0 + int(rows[0]), r0 + int(rows[-1]) + 1, c0 + int(cols[0]), c0 + int(cols[-1]) + 1)
        if TEST_MODE and TEST_WINDOW_CELLS:
            rr, cc = np.nonzero(ROI_MASK[win[0]:win[1], win[2]:win[3]])     # aquifer cell nearest the scene centre
            k = int(np.argmin((rr - (r0 + r1) / 2 + win[0]) ** 2 + (cc - (c0 + c1) / 2 + win[2]) ** 2))
            ra = max(win[0], win[0] + int(rr[k]) - TEST_WINDOW_CELLS // 2)
            ca = max(win[2], win[2] + int(cc[k]) - TEST_WINDOW_CELLS // 2)
            win = (ra, min(win[1], ra + TEST_WINDOW_CELLS), ca, min(win[3], ca + TEST_WINDOW_CELLS))
        out.append(dict(key=f"{p:03d}{w_:03d}", path=p, row=w_, win=win,
                        n_roi=int(ROI_MASK[win[0]:win[1], win[2]:win[3]].sum()),
                        scenes=sorted(scenes, key=lambda r: (r["t"], r["id"]))))
    out.sort(key=lambda P: (-P["n_roi"], P["key"]))
    return out


def window_ctx(key, win):
    """Everything about one path/row window that the maths and the writer need."""
    r0, r1, c0, c1 = win
    mask = np.ascontiguousarray(ROI_MASK[r0:r1, c0:c1])
    return dict(key=key, win=tuple(int(v) for v in win), shape=(r1 - r0, c1 - c0), mask=mask,
                n_roi=int(mask.sum()), dev={})


LATTICE  = 32     # the UTM -> Albers transform is evaluated every 32 pixels and interpolated in between (about 1 mm off)
EDGE_TOL = 0.05   # pixel centres this close (m) to a cell edge are transformed exactly, so that the cell a pixel
                  # falls in is always the one the exact transform gives


def cell_index(xp, r, row0, col0, h, w, W):
    """For every 30 m pixel of a block: the flat index, inside the path/row window, of the 500 m
    cell that holds the pixel's centre; -1 when that cell is outside the window or the aquifer."""
    li, lj = np.arange(0, h + LATTICE, LATTICE), np.arange(0, w + LATTICE, LATTICE)
    lx, ly = transformer(r["crs"]).transform(*np.meshgrid(r["tx"] + (col0 + lj + 0.5) * r["sx"],
                                                         r["ty"] + (row0 + li + 0.5) * r["sy"]))
    lx, ly = xp.asarray(lx), xp.asarray(ly)
    i, j = xp.arange(h), xp.arange(w)
    a, fa = i // LATTICE, ((i % LATTICE) / LATTICE)[:, None]
    b, fb = j // LATTICE, ((j % LATTICE) / LATTICE)[None, :]

    def bilinear(L):
        top = L[a][:, b] * (1 - fb) + L[a][:, b + 1] * fb
        bot = L[a + 1][:, b] * (1 - fb) + L[a + 1][:, b + 1] * fb
        return top * (1 - fa) + bot * fa

    u, v = (bilinear(lx) - GX0) / CELL, (GY0 - bilinear(ly)) / CELL          # column and row, in cells
    near = xp.flatnonzero((xp.abs(u - xp.rint(u)) < EDGE_TOL / CELL) | (xp.abs(v - xp.rint(v)) < EDGE_TOL / CELL))
    if int(near.size):
        k = near.get() if hasattr(near, "get") else near
        ex, ey = transformer(r["crs"]).transform(r["tx"] + (col0 + k % w + 0.5) * r["sx"],
                                                 r["ty"] + (row0 + k // w + 0.5) * r["sy"])
        u[near // w, near % w] = xp.asarray((ex - GX0) / CELL)      # (2-D indexing: u and v need not be C-ordered)
        v[near // w, near % w] = xp.asarray((GY0 - ey) / CELL)
    r0, r1, c0, c1 = W["win"]
    ny, nx = r1 - r0, c1 - c0
    rr = xp.floor(v).astype(xp.int64) - r0
    cc = xp.floor(u).astype(xp.int64) - c0
    inside = (rr >= 0) & (rr < ny) & (cc >= 0) & (cc < nx)
    flat = xp.where(inside, rr * nx + cc, 0)
    name = xp.__name__
    if name not in W["dev"]:                       # the window's aquifer mask, once per device
        W["dev"][name] = xp.asarray(W["mask"].ravel())
    return xp.where(inside & W["dev"][name][flat], flat, -1)


# Earth Engine refuses a pixel request whose size, counted as pixels x bands x 3 bytes, is above 48 MiB.
# Both numbers were read from its own error message and checked against the service: 1448 x 1448 pixels
# of 8 bands (50,320,896 bytes) are served, 1449 x 1448 (50,355,648 bytes) are refused.
EE_REQUEST_BYTES   = 48 * 1024 ** 2
EE_BYTES_PER_VALUE = 3


def max_block_pixels():
    """Most pixels one download block may hold: the BLOCK_PX setting, cut back to Earth Engine's limit."""
    n_bands = len(QA_BANDS) + 6 if ONE_REQUEST_PER_BLOCK else 6
    return max(1, min(int(BLOCK_PX) ** 2, EE_REQUEST_BYTES // (n_bands * EE_BYTES_PER_VALUE)))


def tile_scene(h, w, max_px):
    """Cut an h x w pixel scene into few equal rectangles of at most max_px pixels each; returns
    (row edges, column edges). Rectangles rather than squares: a 7991 x 7251 scene takes 28
    requests this way and 36 as squares, and the time goes into the number of requests. Long thin
    strips are avoided (side ratio at most 2) when that costs no more than a couple of requests,
    because compact blocks are more often skipped whole where the scene leaves the aquifer."""
    cands = []
    for nr in range(1, h + 1):
        bh = -(-h // nr)                         # tallest block with nr rows
        if bh <= max_px:
            nc = -(-w // (max_px // bh))         # fewest columns that keep bh x width within max_px
            bw = -(-w // nc)
            cands.append((nr * nc, max(bh, bw) / min(bh, bw), nr, nc))
            if nc == 1 or nr > 2 * min(c[0] for c in cands):
                break
    fewest = min(c[0] for c in cands)
    compact = [c for c in cands if c[1] <= 2.0 and c[0] <= fewest + max(2, fewest // 8)]
    _, _, nr, nc = min(compact or [c for c in cands if c[0] == fewest])
    return [i * h // nr for i in range(nr + 1)], [j * w // nc for j in range(nc + 1)]


def plan_blocks(r, W):
    """Download blocks of scene r: rectangles of its own 30 m grid that reach an aquifer cell of the window."""
    r0, r1, c0, c1 = W["win"]
    rows, cols = tile_scene(r["h"], r["w"], max_block_pixels())
    blocks = []
    for i in range(len(rows) - 1):
        for j in range(len(cols) - 1):
            row0, col0, h, w = rows[i], cols[j], rows[i + 1] - rows[i], cols[j + 1] - cols[j]
            a, b, c, d = cell_range(r, row0, col0, h, w)
            a, b, c, d = max(a, r0), min(b, r1), max(c, c0), min(d, c1)
            if a < b and c < d and ROI_MASK[a:b, c:d].any():
                blocks.append((row0, col0, h, w))
    return blocks

In [ ]:
# ═══════════════ CELL 6: download raw 30 m blocks to scratch (quality bands first) + check every file ═══════════════
def _grid(r, row0, col0, h, w):
    """A pixel rectangle of scene r on the scene's own grid: nothing is resampled on the way."""
    return dict(dimensions=dict(width=int(w), height=int(h)),
                affineTransform=dict(scaleX=r["sx"], shearX=0, translateX=r["tx"] + col0 * r["sx"],
                                     shearY=0, scaleY=r["sy"], translateY=r["ty"] + row0 * r["sy"]),
                crsCode=r["crs"])


def _check_block(arr, bands, shape):
    """A downloaded block must be exactly what was asked for: these bands, unsigned 16-bit, this shape."""
    if arr.dtype.names != tuple(bands):
        raise ValueError(f"bands {arr.dtype.names} instead of {tuple(bands)}")
    if any(arr.dtype[b] != np.dtype("<u2") for b in bands):
        raise ValueError(f"unexpected data type {arr.dtype}")
    if arr.shape != tuple(shape):
        raise ValueError(f"shape {arr.shape} instead of {tuple(shape)}")
    return arr


_SPLIT_NOTED = [False]


def fetch_array(r, bands, row0, col0, h, w, stop=None):
    """Earth Engine -> memory: one rectangle of scene r as the stored integers (getPixels, so no
    Earth Engine computation is involved). Returns (array, bytes received). A rectangle that Earth
    Engine refuses for its size is fetched as two halves and put together again, so a wrong guess
    about its limit costs a few extra requests and never the run."""
    def once():
        raw = ee.data.getPixels(dict(assetId=r["name"], bandIds=list(bands), grid=_grid(r, row0, col0, h, w),
                                     fileFormat="NPY"))
        return _check_block(np.load(io.BytesIO(raw), allow_pickle=False), bands, (h, w)), len(raw)   # refuse a damaged reply

    try:
        return with_retries(once, f"{r['id']} block {row0},{col0} {bands[0]}..", stop=stop)
    except RequestTooLarge as e:
        if h * w < 4096:
            raise IOError(f"{r['id']}: Earth Engine refuses even a {h} x {w} pixel request ({e})")
        if not _SPLIT_NOTED[0]:
            _SPLIT_NOTED[0] = True
            log.warning("Earth Engine refused a %d x %d pixel request of %d bands for its size (%s). Such blocks "
                        "are fetched in halves from now on; lowering BLOCK_PX avoids the extra requests.",
                        h, w, len(bands), e)
        if h >= w:
            k = h // 2
            top, n1 = fetch_array(r, bands, row0, col0, k, w, stop)
            bottom, n2 = fetch_array(r, bands, row0 + k, col0, h - k, w, stop)
            return np.concatenate([top, bottom], axis=0), n1 + n2
        k = w // 2
        left, n1 = fetch_array(r, bands, row0, col0, h, k, stop)
        right, n2 = fetch_array(r, bands, row0, col0 + k, h, w - k, stop)
        return np.concatenate([left, right], axis=1), n1 + n2


def save_block(path, arr, stop=None):
    """Array -> scratch file, under a temporary name first. Once `stop` is set nothing more is
    written: the scratch folder may already belong to the next run."""
    if stop is not None and stop.is_set():
        raise RunStopping(path)
    tmp = path + ".part"
    with open(tmp, "wb") as f:
        np.save(f, arr, allow_pickle=False)
    os.replace(tmp, path)


def load_block(path, bands, shape):
    """Read a block back from scratch and check it again before it is used."""
    return _check_block(np.load(path, allow_pickle=False), bands, shape)


def qa_usable(xp, qa_pixel, qa_radsat):
    """Pixels that pass the quality rule: present, none of the rejected QA_PIXEL bits, not saturated."""
    ok = ((qa_pixel & 1) == 0) & ((qa_pixel & QA_PIXEL_REJECT) == 0)
    if REJECT_SATURATED:
        ok = ok & (qa_radsat == 0)
    return ok


def download_block(r, blk, W, sdir, stop=None):
    """One block of one scene.

    ONE_REQUEST_PER_BLOCK (default): all eight bands in a single request. Earth Engine needs
    several seconds to answer a request whatever its size, so fewer requests is what makes the
    run faster. The quality bands and the rectangle that bounds the usable pixels go to scratch.
    Otherwise: the two quality bands first, and the six reflective bands only if the block holds
    a usable pixel inside an aquifer cell (less data, twice the requests)."""
    row0, col0, h, w = blk
    tag = f"{row0:05d}_{col0:05d}"
    qa_path = os.path.join(sdir, f"{tag}_qa.npy")
    bands = sorted(set(BAND_LETTERS[SENSOR[r["sat"]]].values()))
    sr_path = os.path.join(sdir, f"{tag}_sr.npy")
    if ONE_REQUEST_PER_BLOCK:
        both, nbytes = fetch_array(r, QA_BANDS + bands, row0, col0, h, w, stop)
        save_block(qa_path, rfn.repack_fields(both[QA_BANDS]), stop)
        ok = (cell_index(np, r, row0, col0, h, w, W) >= 0) & qa_usable(np, both["QA_PIXEL"], both["QA_RADSAT"])
        part = dict(blk=blk, qa=qa_path, sr=None, off=None, bytes=nbytes)
        if ok.any():
            rows, cols = np.flatnonzero(ok.any(axis=1)), np.flatnonzero(ok.any(axis=0))
            a0, a1, b0, b1 = int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1
            save_block(sr_path, rfn.repack_fields(both[bands][a0:a1, b0:b1]), stop)
            part.update(sr=sr_path, off=(a0, a1, b0, b1))
        return part
    qa, nbytes = fetch_array(r, QA_BANDS, row0, col0, h, w, stop)
    save_block(qa_path, qa, stop)
    ok = (cell_index(np, r, row0, col0, h, w, W) >= 0) & qa_usable(np, qa["QA_PIXEL"], qa["QA_RADSAT"])
    if not ok.any():
        return dict(blk=blk, qa=qa_path, sr=None, off=None, bytes=nbytes)
    rows, cols = np.flatnonzero(ok.any(axis=1)), np.flatnonzero(ok.any(axis=0))
    a0, a1, b0, b1 = int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1
    sr, n2 = fetch_array(r, bands, row0 + a0, col0 + b0, a1 - a0, b1 - b0, stop)
    save_block(sr_path, sr, stop)
    return dict(blk=blk, qa=qa_path, sr=sr_path, off=(a0, a1, b0, b1), bytes=nbytes + n2)


def download_group(scenes, W):
    """All blocks of a group of scenes, N_DOWNLOAD_THREADS at a time. Returns
    (ok, permanent, transient): ok = [(scene, parts)], the others = [(scene, reason)].
    A scene counts only when every one of its blocks arrived and passed its check.

    When the run is stopped (stop button, or any error) the requests still waiting are cancelled
    and the ones under way are told to write nothing, so the stop takes effect at once instead of
    after every queued download has finished."""
    plans = {r["id"]: plan_blocks(r, W) for r in scenes}
    need_gb = sum(h * w for b in plans.values() for _, _, h, w in b) * 16 / 1024 ** 3
    ensure_disk_free(f"{W['key']} download", extra_gb=need_gb)
    parts, errors, futs = {r["id"]: [] for r in scenes}, {}, {}
    stop, ex = threading.Event(), ThreadPoolExecutor(N_DOWNLOAD_THREADS)
    try:
        with timed("download"):
            for r in scenes:
                sdir = os.path.join(RAW_DIR, W["key"], r["id"])
                shutil.rmtree(sdir, ignore_errors=True)
                os.makedirs(sdir, exist_ok=True)
                for blk in plans[r["id"]]:
                    futs[ex.submit(download_block, r, blk, W, sdir, stop)] = r
            for fut in as_completed(futs):
                r = futs[fut]
                try:
                    parts[r["id"]].append(fut.result())
                except SceneUnavailable as e:        # only Earth Engine saying "no such scene" is permanent
                    errors[r["id"]] = ("permanent", str(e))
                except Exception as e:
                    errors.setdefault(r["id"], ("transient", f"{type(e).__name__}: {e}"))
    except BaseException:
        stop.set()
        ex.shutdown(wait=False, cancel_futures=True)
        raise
    ex.shutdown(wait=True)
    ok, permanent, transient = [], [], []
    for r in scenes:
        if r["id"] in errors:
            kind, why = errors[r["id"]]
            (permanent if kind == "permanent" else transient).append((r, why))
        else:
            got = sorted(parts[r["id"]], key=lambda p: p["blk"])
            TOTALS["bytes"] = TOTALS.get("bytes", 0) + sum(p["bytes"] for p in got)
            TOTALS["blocks"] = TOTALS.get("blocks", 0) + len(got)
            TOTALS["sr_blocks"] = TOTALS.get("sr_blocks", 0) + sum(1 for p in got if p["sr"])
            ok.append((r, got))
    return ok, permanent, transient


def remove_raw(scenes, key):
    for r in scenes:
        shutil.rmtree(os.path.join(RAW_DIR, key, r["id"]), ignore_errors=True)

In [ ]:
# ═══════════════ CELL 7: index maths on the downloaded blocks, cell means, engine and eemont checks ═══════════════
COUNT_NAMES = ("n_clear", "n_valid")
SCALE = {}        # satellite -> {band: (scale, offset)}, filled from eemont in load_scale_params()
LETTERS_USED = sorted({L for d in INDEX_DEFS.values() for L in d["letters"]})


def load_scale_params():
    """Scale and offset of every band, asked from eemont (its copy of the Earth Engine catalogue)."""
    for sat in SATELLITES:
        ic = ee.ImageCollection(collection_id(sat))
        sc, off = ic.getScaleParams(), ic.getOffsetParams()          # eemont
        bands = BAND_LETTERS[SENSOR[sat]].values()
        if sc is None or off is None or any(b not in sc or b not in off for b in bands):
            raise PipelineStop(f"eemont has no scale/offset for {collection_id(sat)}")
        SCALE[sat] = {b: (float(sc[b]), float(off[b])) for b in bands}
    log.info("Scale/offset from eemont: %s", {s: SCALE[s][BAND_LETTERS[SENSOR[s]]["N"]] for s in SATELLITES})


def _bincount(xp, idx, n, weights=None):
    """Pixels per cell, or the sum of `weights` per cell. An empty selection gives zeros: NumPy's
    bincount does that by itself, CuPy's raises an error instead."""
    if int(idx.size) == 0:
        return xp.zeros(n, dtype=xp.int64 if weights is None else xp.float64)
    return xp.bincount(idx, weights=weights, minlength=n)


def _to_np(a):
    return a.get() if hasattr(a, "get") else np.asarray(a)


def reflectance(xp, dn, sat, dtype):
    """Stored integers -> surface reflectance at the chosen precision (scaling done in 32-bit first:
    16-bit floats cannot hold the integers themselves)."""
    letters = BAND_LETTERS[SENSOR[sat]]
    out = {}
    for L, v in dn.items():
        s, o = SCALE[sat][letters[L]]
        out[L] = (v.astype(xp.float32) * xp.float32(s) + xp.float32(o)).astype(dtype)
    return out


def compute_scene(r, parts, W, xp=None):
    """All blocks of one scene -> one record on the path/row window.

    Every usable 30 m pixel goes to the 500 m cell that holds its centre. A cell's value is the
    plain mean of the index over its usable pixels; n_clear and n_valid count the pixels used and
    the pixels the scene has there. Sums are kept in 64-bit so that averaging adds no error to
    the half-precision pixel values."""
    xp = XP if xp is None else xp
    ny, nx = W["shape"]
    ncell, nidx = ny * nx, len(INDICES)
    letters = BAND_LETTERS[SENSOR[r["sat"]]]
    bands = sorted(set(letters.values()))
    sums = xp.zeros((nidx, ncell), dtype=xp.float64)
    n_clear = xp.zeros(ncell, dtype=xp.int64)
    n_valid = xp.zeros(ncell, dtype=xp.int64)
    dropped = None                                   # per index: pixels whose value was not a finite number
    px = dict(valid=0, clear=0, out_of_range=0, nonfinite=0)
    with np.errstate(all="ignore"):
        for p in parts:
            row0, col0, h, w = p["blk"]
            qa = load_block(p["qa"], QA_BANDS, (h, w))
            flat = cell_index(xp, r, row0, col0, h, w, W)
            qp = xp.asarray(qa["QA_PIXEL"])
            valid = (flat >= 0) & ((qp & 1) == 0)
            n_valid += _bincount(xp, flat[valid], ncell)
            if p["sr"] is None:
                continue
            a0, a1, b0, b1 = p["off"]
            sr = load_block(p["sr"], bands, (a1 - a0, b1 - b0))
            use = (flat[a0:a1, b0:b1] >= 0) & qa_usable(xp, qp[a0:a1, b0:b1], xp.asarray(qa["QA_RADSAT"][a0:a1, b0:b1]))
            dn = {L: xp.asarray(sr[letters[L]]) for L in LETTERS_USED}
            if REQUIRE_VALID_RANGE:
                inr = use
                for L in LETTERS_USED:
                    inr = inr & (dn[L] >= SR_VALID[0]) & (dn[L] <= SR_VALID[1])
                px["out_of_range"] += int((use & ~inr).sum())
                use = inr
            f = flat[a0:a1, b0:b1][use]
            if int(f.size) == 0:
                continue
            n_clear += _bincount(xp, f, ncell)
            refl = reflectance(xp, {L: dn[L][use] for L in LETTERS_USED}, r["sat"], CDTYPE)
            for k, name in enumerate(INDICES):
                v = eval_index(INDEX_DEFS[name], refl, xp)
                bad = ~xp.isfinite(v)
                nbad = int(bad.sum())
                if nbad:                              # a zero denominator: not a number that can be averaged
                    if dropped is None:
                        dropped = xp.zeros((nidx, ncell), dtype=xp.int64)
                    dropped[k] += _bincount(xp, f[bad], ncell)
                    v = xp.where(bad, 0, v)
                    px["nonfinite"] += nbad
                sums[k] += _bincount(xp, f, ncell, weights=v.astype(xp.float32))
        count = n_clear[None, :] if dropped is None else n_clear[None, :] - dropped
        mean = xp.where(count > 0, sums / xp.maximum(count, 1), xp.nan).astype(xp.float32)
        if HALF_PRECISION:
            mean = mean.astype(xp.float16).astype(xp.float32)        # the value stored is a 16-bit float
    mean = _to_np(mean).reshape(nidx, ny, nx)
    n_clear, n_valid = _to_np(n_clear).reshape(ny, nx), _to_np(n_valid).reshape(ny, nx)
    px["valid"], px["clear"] = int(n_valid.sum()), int(n_clear.sum())
    outside = ~W["mask"]
    counts = {}
    for nm, a in (("n_clear", n_clear), ("n_valid", n_valid)):
        a = np.minimum(a, 65534).astype("u2")
        a[outside] = 65535
        counts[nm] = a
    cells = int((n_clear > 0).sum())
    stats = dict(cells_with_value=cells, clear_cell_fraction=cells / max(1, W["n_roi"]),
                 covered_cell_fraction=int((n_valid > 0).sum()) / max(1, W["n_roi"]),
                 overflow=int(np.isinf(mean).sum()), **px)
    return dict(arrays={name: mean[k] for k, name in enumerate(INDICES)}, counts=counts, stats=stats)


# ---------- check 1: the engine at the chosen precision against plain 64-bit NumPy ----------
def _synthetic_scene(folder):
    """A made-up 300 x 300 pixel scene on scratch (random quality flags and reflectances, some fill,
    cloud, saturation and out-of-range values) and a 24 x 24 cell window with a few cells masked out."""
    rng = np.random.default_rng(11)
    n, m = 300, 24
    win = (GRID_NY // 2, GRID_NY // 2 + m, GRID_NX // 2, GRID_NX // 2 + m)
    mask = np.ones((m, m), dtype=bool)
    mask[:3, :5] = False
    W = dict(key="selftest", win=win, shape=(m, m), mask=mask, n_roi=int(mask.sum()), dev={})
    ux, uy = pyproj.Transformer.from_crs(GRID_CRS, "EPSG:32614", always_xy=True).transform(
        GX0 + (win[2] + m / 2) * CELL, GY0 - (win[0] + m / 2) * CELL)
    r = dict(id="selftest", sat="LC08", crs="EPSG:32614", sx=30.0, sy=-30.0, w=n, h=n,
             tx=round(ux / 30) * 30.0 - n * 15.0, ty=round(uy / 30) * 30.0 + n * 15.0)
    qa = np.zeros((n, n), dtype=[(b, "<u2") for b in QA_BANDS])
    u = rng.random((n, n))
    qa["QA_PIXEL"] = np.where(u < 0.05, 1, np.where(u < 0.25, 21824 | 8, 21824))     # fill, cloud, clear
    qa["QA_RADSAT"] = (rng.random((n, n)) < 0.01) * 2
    a0, a1, b0, b1 = 10, 290, 5, 300
    bands = sorted(set(BAND_LETTERS["OLI"].values()))
    sr = np.zeros((a1 - a0, b1 - b0), dtype=[(b, "<u2") for b in bands])
    for b in bands:
        sr[b] = rng.integers(7000, 20000, sr.shape)
    os.makedirs(folder, exist_ok=True)
    qa_path, sr_path = os.path.join(folder, "qa.npy"), os.path.join(folder, "sr.npy")
    np.save(qa_path, qa)
    np.save(sr_path, sr)
    nothing = np.zeros((n, n), dtype=qa.dtype)        # a block that is all fill: no pixel is selected from it,
    nothing["QA_PIXEL"] = 1                           # the case in which CuPy and NumPy behave differently
    none_path = os.path.join(folder, "qa_fill.npy")
    np.save(none_path, nothing)
    return r, [dict(blk=(0, 0, n, n), qa=qa_path, sr=sr_path, off=(a0, a1, b0, b1)),
               dict(blk=(0, 0, n, n), qa=none_path, sr=None, off=None)], W


def engine_self_test():
    """Two tests on the active engine and precision. (1) Random reflectances through every formula,
    compared with 64-bit NumPy. (2) A made-up scene through the whole chain (cell assignment, pixel
    rule, maths, averaging); on a GPU the result must match the CPU's. A GPU that fails either test
    is dropped in favour of the CPU instead of stopping the run."""
    global XP, ENGINE
    rng = np.random.default_rng(7)
    ref = {L: rng.uniform(0.02, 0.6, 200_000) for L in LETTERS}
    for attempt in (0, 1):
        try:
            dev = {L: XP.asarray(v.astype(CDTYPE)) for L, v in ref.items()}
            worst = {}
            with np.errstate(all="ignore"):
                for name in INDICES:
                    want = eval_index(INDEX_DEFS[name], ref, np)
                    got = _to_np(eval_index(INDEX_DEFS[name], dev, XP)).astype("f8")
                    if got.dtype != np.float64 or got.shape != want.shape:
                        raise DataIntegrityError(f"{name}: unexpected result {got.dtype} {got.shape}")
                    err = np.abs(got - want) / (1 + np.abs(want))
                    worst[name] = float(np.nanmedian(err))
            limit = 3e-3 if HALF_PRECISION else 1e-5
            bad = {k: v for k, v in worst.items() if not v <= limit}
            if bad:
                raise DataIntegrityError(f"median relative error above {limit}: {bad}")
            folder = os.path.join(RAW_DIR, "_selftest")
            try:
                r, parts, W = _synthetic_scene(folder)
                cpu = compute_scene(r, parts, W, xp=np)
                if cpu["stats"]["cells_with_value"] == 0 or cpu["stats"]["out_of_range"] == 0:
                    raise DataIntegrityError(f"synthetic scene gave an implausible result: {cpu['stats']}")
                if XP is not np:
                    gpu = compute_scene(r, parts, W, xp=XP)
                    for nm in COUNT_NAMES:
                        if not np.array_equal(cpu["counts"][nm], gpu["counts"][nm]):
                            raise DataIntegrityError(f"{nm} differs between GPU and CPU")
                    for nm in INDICES:
                        a, b = cpu["arrays"][nm], gpu["arrays"][nm]
                        if (not np.array_equal(np.isnan(a), np.isnan(b))
                                or np.nanmax(np.abs(a - b) / (1 + np.abs(a))) > limit):
                            raise DataIntegrityError(f"{nm} differs between GPU and CPU")
            finally:
                shutil.rmtree(folder, ignore_errors=True)
            log.info("Engine self-test OK on %s: formulas within %.1e .. %.1e of 64-bit (median, %s); whole chain "
                     "on a synthetic scene OK%s", ENGINE, min(worst.values()), max(worst.values()),
                     np.dtype(CDTYPE).name, "" if XP is np else ", GPU = CPU")
            return worst
        except Exception as e:
            if XP is np or attempt:
                raise
            log.warning("GPU engine failed its self-test (%s) - continuing on the CPU", e)
            XP, ENGINE = np, f"CPU (NumPy {np.__version__}, GPU self-test failed)"


# ---------- check 2: the same pixels through eemont on Earth Engine's servers ----------
def eemont_cross_check(r):
    """A small window of scene r, computed twice: by eemont on Earth Engine (scaleAndOffset +
    spectralIndices, 64-bit) and by this notebook from the downloaded integers. Run in 64-bit the
    two must agree to rounding, which proves the formulas, band letters, scale and offset are
    eemont's. The same comparison at the working precision shows what half precision costs."""
    n = EEMONT_CHECK_PX
    row0, col0 = (r["h"] - n) // 2, (r["w"] - n) // 2
    letters = BAND_LETTERS[SENSOR[r["sat"]]]
    bands = sorted(set(letters.values()))
    g = _grid(r, row0, col0, n, n)
    raw = with_retries(lambda: ee.data.getPixels(dict(assetId=r["name"], bandIds=bands, grid=g,
                                                      fileFormat="NUMPY_NDARRAY")), "cross-check pixels")
    present = np.all([raw[b] != 0 for b in bands], axis=0)
    if not present.any():
        log.warning("Cross-check window of %s holds no data - skipped", r["id"])
        return None
    dn = {L: raw[letters[L]][present] for L in LETTERS}
    scaled = ee.Image(f"{collection_id(r['sat'])}/{r['id']}").scaleAndOffset()                  # eemont
    server = {}
    by_params = {}
    for name in INDICES:
        by_params.setdefault(json.dumps(INDEX_DEFS[name]["params"], sort_keys=True), []).append(name)
    for names in by_params.values():
        kw = dict(INDEX_PARAMS, **INDEX_DEFS[names[0]]["params"])
        img = scaled.spectralIndices(names, G=kw["g"], C1=kw["C1"], C2=kw["C2"], L=kw["L"]).select(names)  # eemont
        got = with_retries(lambda: ee.data.computePixels(dict(expression=img.toDouble(), grid=g,
                                                              fileFormat="NUMPY_NDARRAY")), "cross-check eemont")
        for name in names:
            server[name] = got[name][present].astype("f8")
    r64 = {L: dn[L].astype("f8") * SCALE[r["sat"]][letters[L]][0] + SCALE[r["sat"]][letters[L]][1] for L in LETTERS}
    rwk = reflectance(np, dn, r["sat"], CDTYPE)
    worst64, worstwk = 0.0, {}
    with np.errstate(all="ignore"):
        for name in INDICES:
            d = INDEX_DEFS[name]
            cat = dict(d, code=_compile_formula(d["catalogue_formula"])[0])      # eemont's own formula text
            ok = np.isfinite(server[name])
            e64 = np.abs(eval_index(cat, r64, np) - server[name])[ok] / (1 + np.abs(server[name][ok]))
            if e64.size and e64.max() > 1e-9:
                raise DataIntegrityError(f"{name} on {r['id']}: local 64-bit result differs from eemont on Earth "
                                         f"Engine by {e64.max():.2e} (relative). Formula, band letters or scaling "
                                         "do not match eemont.")
            worst64 = max(worst64, float(e64.max()) if e64.size else 0.0)
            ewk = np.abs(eval_index(cat, rwk, np).astype("f8") - server[name])[ok]
            worstwk[name] = float(np.percentile(ewk, 99)) if ewk.size else 0.0
    log.info("eemont cross-check OK on %s (%s, %d px): 64-bit agreement %.1e; per-pixel error at %s, 99th "
             "percentile: %s", r["id"], SAT_NAME[r["sat"]], int(present.sum()), worst64, np.dtype(CDTYPE).name,
             ", ".join(f"{k} {v:.1e}" for k, v in sorted(worstwk.items(), key=lambda kv: -kv[1])[:4]))
    return dict(scene=r["id"], agreement_64bit=worst64, p99_error_working_precision=worstwk)

In [ ]:
# ═══════════════ CELL 8: CF NetCDF writer, checkpoints, resume state, read-back check, file audit ═══════════════
COUNT_VARS = ["n_clear", "n_valid"]
SCENE_VARS = ["satellite", "cloud_cover", "sun_elevation", "clear_cell_fraction", "covered_cell_fraction"]
SCENE_TEXT = ["source_scene", "source_product_id", "source_asset_updated"]
DRIVE_RETRY_S = 20                       # pause before a file on Drive is read again
SETTINGS_SIGNATURE = json.dumps(dict(
    indices=INDICES, formulas={k: INDEX_DEFS[k]["formula"] for k in INDICES},
    params={k: INDEX_DEFS[k]["params"] for k in INDICES}, qa_reject=QA_PIXEL_REJECT, saturated=REJECT_SATURATED,
    valid_range=REQUIRE_VALID_RANGE, half=HALF_PRECISION, tier=TIER), sort_keys=True)


def pr_filename(key):
    return f"{PRODUCT_TAG}_p{key[:3]}r{key[3:]}_{ROI_LABEL}.nc"


def _cf_attr(v):
    if isinstance(v, (str, int, float, np.number, np.ndarray)):
        return v
    if isinstance(v, (list, tuple)):
        return np.asarray(v)
    return str(v)


def pixel_rule_text():
    bits = {1: "dilated cloud", 2: "cirrus", 3: "cloud", 4: "cloud shadow", 5: "snow"}
    rule = "fill, " + ", ".join(v for b, v in bits.items() if QA_PIXEL_REJECT >> b & 1) + " (QA_PIXEL)"
    if REJECT_SATURATED:
        rule += "; any band saturated (QA_RADSAT)"
    if REQUIRE_VALID_RANGE:
        rule += f"; a reflective band outside USGS's valid range {SR_VALID[0]}-{SR_VALID[1]}"
    return rule


def create_pr_nc(path, P, W, checkpoint=None):
    """Create the empty CF file of one path/row on its window of the common 500 m grid. Written
    under a temporary name and renamed when complete.

    checkpoint = dict(index, previous): the same file layout for one group of scenes of a path/row
    that is still being processed (no lat/lon, to keep it small). `previous` is the resume point
    the group started from, which is how a missing checkpoint is noticed later."""
    r0, r1, c0, c1 = W["win"]
    ny, nx = W["shape"]
    crs = pyproj.CRS.from_user_input(GRID_CRS)
    xs = GX0 + (c0 + np.arange(nx) + 0.5) * CELL                 # cell-centre coordinates
    ys = GY0 - (r0 + np.arange(ny) + 0.5) * CELL
    zkw = dict(zlib=COMPRESS_LEVEL > 0, complevel=max(COMPRESS_LEVEL, 1), shuffle=True)
    chunks = (1, min(CHUNK_YX, ny), min(CHUNK_YX, nx))
    precision = ("IEEE 754 half precision (16-bit) values held in a 32-bit float variable; netCDF has no "
                 "16-bit float type. About 3 significant digits." if HALF_PRECISION else "32-bit float")
    tmp = path + ".new"
    ds = nc4.Dataset(tmp, "w", format="NETCDF4")
    try:
        ds.createDimension("time", None)
        ds.createDimension("y", ny)
        ds.createDimension("x", nx)
        tv = ds.createVariable("time", "f8", ("time",), chunksizes=(512,))
        tv.units, tv.calendar, tv.standard_name, tv.axis = TIME_UNITS, TIME_CALENDAR, "time", "T"
        tv.long_name = "scene start time (UTC)"
        yv = ds.createVariable("y", "f8", ("y",))
        yv.standard_name, yv.units, yv.axis = "projection_y_coordinate", "m", "Y"
        yv.long_name = "northing of cell centre"
        yv[:] = ys
        xv = ds.createVariable("x", "f8", ("x",))
        xv.standard_name, xv.units, xv.axis = "projection_x_coordinate", "m", "X"
        xv.long_name = "easting of cell centre"
        xv[:] = xs
        coords = None
        if WRITE_LATLON and checkpoint is None:
            X, Y = np.meshgrid(xs, ys)
            lon, lat = pyproj.Transformer.from_crs(crs, 4326, always_xy=True).transform(X, Y)
            for nm, std, units, val in (("lat", "latitude", "degrees_north", lat), ("lon", "longitude", "degrees_east", lon)):
                v = ds.createVariable(nm, "f8", ("y", "x"), **zkw)
                v.standard_name, v.units, v.long_name = std, units, f"{std} of cell centre (WGS 84)"
                v[:] = val
            coords = "lat lon"
        gm = ds.createVariable("crs", "i4")
        for k, v in crs.to_cf().items():
            gm.setncattr(k, _cf_attr(v))
        gm.spatial_ref = crs.to_wkt()
        gm.GeoTransform = f"{GX0 + c0 * CELL} {CELL} 0.0 {GY0 - r0 * CELL} 0.0 {-CELL}"   # GDAL order, this window
        gm.grid_origin = np.array([GX0, GY0], dtype="f8")          # upper-left corner of the whole 500 m grid
        gm.grid_shape = np.array([GRID_NY, GRID_NX], dtype="i4")
        gm.grid_window = np.array(W["win"], dtype="i4")            # row0, row1, col0, col1 stored in this file
        gm.long_name = "USGS High Plains 500 m grid (NAD83 / Conus Albers)"

        mv = ds.createVariable("roi_mask", "i1", ("y", "x"), **zkw)
        mv.long_name = f"1 where the cell centre lies inside the {ROI_LABEL} aquifer polygon"
        mv.flag_values = np.array([0, 1], dtype="i1")
        mv.flag_meanings = "outside_roi inside_roi"
        mv.grid_mapping = "crs"
        if coords:
            mv.coordinates = coords
        mv[:] = W["mask"].astype("i1")

        for name in INDICES:
            d = INDEX_DEFS[name]
            v = ds.createVariable(name, "f4", ("time", "y", "x"), fill_value=np.float32(np.nan), chunksizes=chunks, **zkw)
            v.long_name = d["long_name"]
            v.units = "1"
            v.grid_mapping = "crs"
            if coords:
                v.coordinates = coords
            v.cell_methods = "area: mean (over the usable 30 m pixels whose centre lies in the cell)"
            v.ancillary_variables = " ".join(COUNT_VARS)
            v.formula = d["formula"]
            v.formula_source = d["origin"]
            v.formula_bands = "B: blue, G: green, R: red, N: near infrared, S1: shortwave infrared 1, S2: shortwave infrared 2"
            if d["params"]:
                v.formula_parameters = json.dumps(d["params"], sort_keys=True)
            if d["kernels"]:
                v.formula_kernel = "k(a, b) = exp(-(a - b)^2 / (2 sigma^2)), sigma = 0.5 (a + b)  [eemont default, RBF]"
            v.references = d["reference"]
            v.application_domain = d["domain"]
            v.stored_precision = precision
            v.comment = ("Computed per 30 m pixel from Landsat Collection 2 Level-2 surface reflectance, then "
                         "averaged per cell. Pixels left out: " + pixel_rule_text() + ". Cells outside the "
                         "polygon and cells without a usable pixel are _FillValue.")
        for nm, text in (("n_clear", "number of 30 m pixels averaged in the cell"),
                         ("n_valid", "number of 30 m pixels of the scene in the cell (clear or not)")):
            v = ds.createVariable(nm, "u2", ("time", "y", "x"), fill_value=np.uint16(65535), chunksizes=chunks, **zkw)
            v.long_name, v.units, v.grid_mapping = text, "1", "crs"
            if coords:
                v.coordinates = coords
            v.comment = "A full cell holds about 278 pixels. _FillValue marks cells outside the polygon."

        sv = ds.createVariable("satellite", "i1", ("time",), chunksizes=(512,))
        sv.long_name = "Landsat satellite number"
        sv.flag_values = np.array(sorted(set(SAT_NUMBER.values())), dtype="i1")
        sv.flag_meanings = "Landsat_4_TM Landsat_5_TM Landsat_7_ETM Landsat_8_OLI Landsat_9_OLI2"
        for nm, text, units in (("cloud_cover", "scene cloud cover reported by USGS (CLOUD_COVER)", "percent"),
                                ("sun_elevation", "sun elevation at scene centre (SUN_ELEVATION)", "degree"),
                                ("clear_cell_fraction", "share of the aquifer cells of this file that hold a value", "1"),
                                ("covered_cell_fraction", "share of the aquifer cells of this file the scene covers", "1")):
            v = ds.createVariable(nm, "f4", ("time",), chunksizes=(512,))
            v.long_name, v.units = text, units
        for nm, text in (("source_scene", "Earth Engine scene id of each record"),
                         ("source_product_id", "USGS LANDSAT_PRODUCT_ID of each record"),
                         ("source_asset_updated", "time the Earth Engine copy of the scene was last updated")):
            ds.createVariable(nm, str, ("time",)).long_name = text

        ds.Conventions = "CF-1.9"
        ds.title = (f"Landsat Collection 2 Level-2 vegetation and water indices, {ROI_LABEL} aquifer, WRS-2 path "
                    f"{P['path']} row {P['row']}, every scene, 500 m cell means")
        ds.summary = ("One record per Landsat scene. Each index is computed per 30 m pixel from surface "
                      "reflectance and averaged over the usable pixels whose centre lies in each cell of the USGS "
                      "High Plains 500 m grid. Scenes are not composited, harmonised between sensors or gap-filled; "
                      "Landsat 7 scan-line gaps stay gaps.")
        ds.institution = "U.S. Geological Survey (source data); packaged by the MSU groundwater project"
        ds.source = ("USGS Landsat Collection 2 Level-2 Tier 1 surface reflectance, read as stored integers from "
                     "Google Earth Engine (" + ", ".join(collection_id(s) for s in SATELLITES) + "); index formulas, "
                     f"scale and offset from eemont {eemont.__version__}")
        ds.references = ("Landsat Collection 2 Level-2 Science Products: " + ", ".join(
            f"https://doi.org/{d}" for d in sorted(set(SAT_DOI.values()))) + "; eemont: https://doi.org/10.21105/joss.03168; "
            "Awesome Spectral Indices: https://doi.org/10.1038/s41597-023-02096-0")
        ds.history = f"{iso(utcnow())} created by Landsat_Indices_Ogallala.ipynb"
        ds.wrs_path, ds.wrs_row, ds.roi = np.int32(P["path"]), np.int32(P["row"]), ROI_LABEL
        ds.indices = " ".join(INDICES)
        ds.pixels_left_out = pixel_rule_text()
        ds.settings_signature = SETTINGS_SIGNATURE
        inside = W["mask"]
        if coords:
            ds.geospatial_lat_min, ds.geospatial_lat_max = float(lat[inside].min()), float(lat[inside].max())
            ds.geospatial_lon_min, ds.geospatial_lon_max = float(lon[inside].min()), float(lon[inside].max())
        ds.processing_status = "in_progress"
        ds.resume_after = ""
        ds.resume_after_seconds = np.float64("nan")
        ds.committed_records = np.int32(0)
        if checkpoint is not None:
            prev = checkpoint["previous"]
            ds.processing_status = "checkpoint"
            ds.checkpoint_index = np.int32(checkpoint["index"])
            ds.previous_resume_after_seconds = np.float64("nan" if prev is None else prev)
            ds.comment = ("One group of scenes of a path/row that is still being processed. The notebook joins "
                          "the checkpoints, in order, into the path/row's own file when its last scene is done.")
    finally:
        ds.close()
    os.replace(tmp, path)


def _state(path):
    with nc4.Dataset(path, "r") as ds:
        ds.set_auto_maskandscale(False)
        n = len(ds.variables["time"])
        committed = int(getattr(ds, "committed_records", -1))
        if committed != n:
            return None, f"holds {n} records but its last commit says {committed} (an interrupted write)"
        t = np.asarray(ds.variables["time"][:], dtype="f8")
        if n > 1 and not np.all(np.diff(t) > 0):
            return None, "has a time axis that is not strictly increasing"
        win = tuple(int(v) for v in ds.variables["crs"].grid_window)
        mask = np.asarray(ds.variables["roi_mask"][:]).astype(bool)
        ra = float(getattr(ds, "resume_after_seconds", float("nan")))
        pv = float(getattr(ds, "previous_resume_after_seconds", float("nan")))
        return dict(n=n, status=str(getattr(ds, "processing_status", "in_progress")),
                    resume_after=None if math.isnan(ra) else ra, previous=None if math.isnan(pv) else pv,
                    first=float(t[0]) if n else None, last=float(t[-1]) if n else None, window=win,
                    signature=str(getattr(ds, "settings_signature", "")),
                    mask_ok=bool(np.array_equal(mask, ROI_MASK[win[0]:win[1], win[2]:win[3]])),
                    size=os.path.getsize(path)), None


def read_state(path, tries=1):
    """(state, None) of a path/row file or a checkpoint, or (None, why) when it cannot be used:
    unreadable, a record count that disagrees with the last commit, or a time axis that is not
    increasing. From Drive it is tried several times: a Drive that is slow or only just mounted
    must never be mistaken for a damaged file."""
    why = None
    for attempt in range(max(1, tries)):
        if attempt:
            time.sleep(DRIVE_RETRY_S * attempt)
        try:
            state, why = _state(path)
            if state is not None:
                return state, None
        except Exception as e:
            why = f"cannot be read ({type(e).__name__}: {str(e)[:160]})"
        log.warning("%s %s (attempt %d of %d)", os.path.basename(path), why, attempt + 1, max(1, tries))
    return None, why


def scene_seconds(r):
    return (parse_iso(r["t"]) - EPOCH).total_seconds()


class PRWriter:
    """Single writer of one path/row file. Records are appended strictly in time order; commit()
    stores the resume point and flushes, so the file on disk is always self-describing."""

    def __init__(self, path, cursor=None):
        self.path, self.ds, self.cursor = path, None, cursor
        self.open()

    def open(self):
        self.ds = nc4.Dataset(self.path, "a")
        self.ds.set_auto_maskandscale(False)
        tv = self.ds.variables["time"]
        self.n = len(tv)
        self.last = float(tv[self.n - 1]) if self.n else -math.inf

    def append(self, r, res):
        """Write one scene. Returns False (nothing written) for a duplicate or out-of-order time."""
        tnum = scene_seconds(r)
        if tnum <= self.last:
            return False
        i, v = self.n, self.ds.variables
        for name in INDICES:
            v[name][i, :, :] = res["arrays"][name]
        for name in COUNT_VARS:
            v[name][i, :, :] = res["counts"][name]
        v["satellite"][i] = SAT_NUMBER[r["sat"]]
        v["cloud_cover"][i] = r["cloud"]
        v["sun_elevation"][i] = r["sun_elev"]
        v["clear_cell_fraction"][i] = res["stats"]["clear_cell_fraction"]
        v["covered_cell_fraction"][i] = res["stats"]["covered_cell_fraction"]
        v["source_scene"][i] = r["id"]
        v["source_product_id"][i] = r["product"]
        v["source_asset_updated"][i] = r["updated"]
        v["time"][i] = tnum
        self.n, self.last = i + 1, tnum
        return True

    def commit(self, status=None):
        ds = self.ds
        has = self.cursor is not None
        ds.resume_after = iso(EPOCH + timedelta(seconds=self.cursor)) if has else ""   # last scene handled
        ds.resume_after_seconds = np.float64(self.cursor if has else "nan")            # the same, exact
        ds.committed_records = np.int32(self.n)
        if status:
            ds.processing_status = status
        if self.n:
            tv = ds.variables["time"]
            ds.time_coverage_start = iso(EPOCH + timedelta(seconds=float(tv[0])))
            ds.time_coverage_end = iso(EPOCH + timedelta(seconds=float(tv[self.n - 1])))
        ds.sync()

    def close(self):
        if self.ds is not None:
            self.ds.close()
            self.ds = None


def open_checkpoint(path, P, W, index, previous):
    """A new, empty checkpoint on scratch for the group of scenes that follows `previous`."""
    create_pr_nc(path, P, W, checkpoint=dict(index=index, previous=previous))
    return PRWriter(path, previous)


def copy_from_drive(src, dst):
    """Size-checked copy of a checkpoint from Drive to scratch, tried several times; a Drive that
    will not hand the file over stops the run cleanly."""
    why = None
    for attempt in range(max(1, DRIVE_READ_TRIES)):
        if attempt:
            time.sleep(DRIVE_RETRY_S * attempt)
        try:
            size = os.path.getsize(src)
            ensure_disk_free("checkpoint from Drive", extra_gb=size / 1024 ** 3)
            tmp = dst + ".part"
            shutil.copyfile(src, tmp)
            if os.path.getsize(tmp) != size:
                raise IOError("the copy is incomplete")
            os.replace(tmp, dst)
            return size
        except PipelineStop:
            raise
        except Exception as e:
            why = f"{type(e).__name__}: {str(e)[:160]}"
    raise PipelineStop(f"{os.path.basename(src)} could not be copied from Drive ({why}). Nothing was changed - "
                       "run again in a few minutes.")


def _raw_bytes(a):
    return np.ascontiguousarray(a).tobytes()


def append_records(writer, src_path):
    """Copy every record of a checkpoint to the end of the file being put together, as stored
    (nothing is decoded, rounded or re-scaled on the way). Returns the number of records."""
    with nc4.Dataset(src_path, "r") as src:
        src.set_auto_maskandscale(False)
        t = np.asarray(src.variables["time"][:], dtype="f8")
        v = writer.ds.variables
        for i in range(len(t)):
            if not t[i] > writer.last:
                raise DataIntegrityError(f"{os.path.basename(src_path)} record {i} is not later than the record "
                                         "before it in the path/row's file")
            j = writer.n
            for name in INDICES + COUNT_VARS:
                v[name][j, :, :] = src.variables[name][i, :, :]
            for name in SCENE_VARS:
                v[name][j] = src.variables[name][i]
            for name in SCENE_TEXT:
                v[name][j] = str(src.variables[name][i])
            v["time"][j] = t[i]
            writer.n, writer.last = j + 1, float(t[i])
    return len(t)


def verify_merge(path, parts):
    """Every record of every checkpoint must be in the path/row's file, bit for bit. Returns the
    number of records compared; raises DataIntegrityError on the first difference."""
    total = 0
    with nc4.Dataset(path, "r") as ds:
        ds.set_auto_maskandscale(False)
        t = np.asarray(ds.variables["time"][:], dtype="f8")
        for part in parts:
            with nc4.Dataset(part, "r") as src:
                src.set_auto_maskandscale(False)
                ts = np.asarray(src.variables["time"][:], dtype="f8")
                for i, tv in enumerate(ts):
                    j = int(np.searchsorted(t, tv))
                    if j >= len(t) or t[j] != tv:
                        raise DataIntegrityError(f"{os.path.basename(part)} record {i} is missing from {os.path.basename(path)}")
                    for name in INDICES + COUNT_VARS + SCENE_VARS:
                        a, b = np.asarray(src.variables[name][i]), np.asarray(ds.variables[name][j])
                        if a.dtype != b.dtype or a.shape != b.shape or _raw_bytes(a) != _raw_bytes(b):
                            raise DataIntegrityError(f"{os.path.basename(path)} record {j} variable {name} differs "
                                                     f"from {os.path.basename(part)} record {i}")
                    for name in SCENE_TEXT:
                        if str(src.variables[name][i]) != str(ds.variables[name][j]):
                            raise DataIntegrityError(f"{os.path.basename(path)} record {j} {name} differs from "
                                                     f"{os.path.basename(part)} record {i}")
                total += len(ts)
    return total


def readback_check(writer, index, res, W):
    """Re-read record `index` from the .nc and compare it bit for bit with what was computed."""
    v = writer.ds.variables
    for name in INDICES + COUNT_VARS:
        want = res["arrays"][name] if name in res["arrays"] else res["counts"][name]
        got = np.asarray(v[name][index, :, :])
        if got.dtype != want.dtype or not np.array_equal(got.view(f"u{got.dtype.itemsize}"),
                                                         want.view(f"u{want.dtype.itemsize}")):
            raise DataIntegrityError(f"{W['key']} record {index} variable {name}: the file differs from memory")
    if HALF_PRECISION:
        a = res["arrays"][INDICES[0]]
        if not np.array_equal(a.astype("f2").astype("f4"), a, equal_nan=True):
            raise DataIntegrityError(f"{W['key']} record {index}: values are not half-precision numbers")


def audit_nc(path):
    """Read-only structural audit of a path/row file. Returns (ok, problems, info)."""
    problems, info = [], {}
    try:
        with nc4.Dataset(path, "r") as ds:
            ds.set_auto_maskandscale(False)
            if not str(getattr(ds, "Conventions", "")).startswith("CF-"):
                problems.append("no CF Conventions attribute")
            names = str(getattr(ds, "indices", "")).split()
            need = ["time", "y", "x", "crs", "roi_mask", "satellite", "source_scene"] + names + COUNT_VARS
            missing = [v for v in need if v not in ds.variables]
            if missing or not names:
                return False, [f"missing variables {missing}"], info
            n = len(ds.variables["time"])
            t = np.asarray(ds.variables["time"][:], dtype="f8")
            if int(getattr(ds, "committed_records", -1)) != n:
                problems.append("record count differs from committed_records")
            if n > 1 and not np.all(np.diff(t) > 0):
                problems.append("time axis not strictly increasing")
            inside = np.asarray(ds.variables["roi_mask"][:]).astype(bool)
            if not inside.any():
                problems.append("roi_mask is empty")
            for name in names + COUNT_VARS:
                v = ds.variables[name]
                if v.dimensions != ("time", "y", "x") or getattr(v, "grid_mapping", None) != "crs":
                    problems.append(f"{name}: unexpected dimensions or no grid_mapping")
            for i in sorted({0, n // 2, n - 1}) if n else []:
                a = np.asarray(ds.variables[names[0]][i, :, :])
                c = np.asarray(ds.variables["n_clear"][i, :, :])
                if np.count_nonzero(~np.isnan(a[~inside])) or np.count_nonzero(c[~inside] != 65535):
                    problems.append(f"record {i}: values outside the polygon")
                if not np.array_equal(~np.isnan(a[inside]), c[inside] > 0):
                    problems.append(f"record {i}: {names[0]} and n_clear disagree on which cells hold a value")
            if n:
                info.update(first=iso(EPOCH + timedelta(seconds=float(t[0])))[:10],
                            last=iso(EPOCH + timedelta(seconds=float(t[-1])))[:10])
            info.update(records=n, status=str(getattr(ds, "processing_status", "?")),
                        window=f"{inside.shape[0]}x{inside.shape[1]}", size=os.path.getsize(path))
    except Exception as e:
        problems.append(f"cannot be opened/read: {type(e).__name__}: {e}")
    return (not problems), problems, info

In [ ]:
# ═══════════════ CELL 9: path/row by path/row pipeline, one checkpoint on Drive per group of scenes, run ═══════════════
# How a run keeps its place. Every finished group of scenes is saved to Drive at once, as a small file
#   <output folder>/_checkpoints/p<path>r<row>/checkpoint_00001.nc, checkpoint_00002.nc, ...
# and never touched again. Each checkpoint records the resume point it started from and the one it reached.
# A new run reads only Drive: the path/row's file (if it has one) followed by its checkpoints, each of which
# must start exactly where the one before it ended. Nothing on scratch is trusted. When the last scene of a
# path/row is done the checkpoints are joined, bit for bit, into the path/row's single file.
_CKPT_RE  = re.compile(r"^checkpoint_(\d{5})\.nc$")
_ASIDE_RE = re.compile(r"_p(\d{3})r(\d{3})_.*\.nc\.unreadable_")
MAX_ORPHAN_CHECKPOINTS = 5      # more checkpoints than this after a gap: stop and say so instead of redoing them
FINISHED = {}                   # path/row -> when its file was uploaded in this session
LAST = {}                       # where the run stands (for the message printed when it stops)


def date_of(seconds):
    return "the start" if seconds is None else iso(EPOCH + timedelta(seconds=seconds))[:10]


def ckpt_dir(key, drive=True):
    return os.path.join(DRIVE_OUT if drive else NC_DIR, CKPT_FOLDER, f"p{key[:3]}r{key[3:]}")


def list_checkpoints(key):
    """[(index, path)] of the checkpoints of a path/row on Drive, in order."""
    d = ckpt_dir(key)
    if not os.path.isdir(d):
        return []
    return sorted((int(m.group(1)), os.path.join(d, name)) for name in os.listdir(d)
                  for m in [_CKPT_RE.match(name)] if m)


def save_to_drive(src, dst, key, what):
    """Verified copy to Drive; a Drive that will not take the file stops the run cleanly."""
    try:
        robust_drive_copy(src, dst)
    except PipelineStop:
        raise
    except Exception as e:
        raise PipelineStop(f"[{key}] {what} could not be saved to Drive ({e}). Everything saved before it is "
                           "still there - re-run to resume.")


class Progress:
    def __init__(self, total):
        self.total, self.done, self.run, self.t0 = total, 0, 0, time.time()

    def add(self, n, resumed=False):
        self.done += n
        if not resumed:
            self.run += n

    def line(self):
        rate = self.run / max(time.time() - self.t0, 1e-9)
        eta = (self.total - self.done) / rate / 3600 if rate > 0 else float("nan")
        return (f"{self.done}/{self.total} scenes ({100 * self.done / max(1, self.total):.1f} %), "
                f"{rate * 60:.1f}/min, ETA {eta:.1f} h")


def _compute_safely(job):
    r, parts, W = job
    try:
        return compute_scene(r, parts, W)
    except Exception as e:                       # a block that went bad on scratch: fetch the scene again
        return dict(error=f"{type(e).__name__}: {e}")


# ---------- resume: what Drive holds for a path/row ----------
def _stop_unusable(key, path, why):
    raise PipelineStop(
        f"[{key}] {os.path.relpath(path, DRIVE_OUT)} is on Drive but {why}. Nothing was changed or deleted. Run "
        "again in a few minutes (Drive can be slow just after it is mounted). This notebook never starts a "
        "path/row over on its own: if the file really is damaged, move it out of the folder yourself and run again.")


def _same_settings(key, path, st):
    if st["signature"] != SETTINGS_SIGNATURE:
        raise PipelineStop(f"[{key}] {os.path.basename(path)} was made with other settings (indices, formulas, pixel "
                           "rule or precision). Restore those settings in Cell 2, or move this path/row's file and "
                           "its checkpoint folder out of the output folder to start it again.")
    if not st["mask_ok"]:
        raise PipelineStop(f"[{key}] {os.path.basename(path)}: its aquifer mask differs from the shapefile's.")


def resume_plan(P):
    """Where a path/row continues: the end of its file on Drive (if it has one) plus the unbroken
    chain of checkpoints that follows. A file that cannot be read stops the run; it is never a
    reason to start again."""
    key = P["key"]
    drive_nc = os.path.join(DRIVE_OUT, pr_filename(key))
    base = None
    if os.path.exists(drive_nc):
        base, why = read_state(drive_nc, DRIVE_READ_TRIES)
        if base is None:
            _stop_unusable(key, drive_nc, why)
        _same_settings(key, drive_nc, base)
    cursor = base["resume_after"] if base else None
    if base and base["n"] and (cursor is None or base["last"] > cursor):
        # A file the first version of this notebook left when it was stopped in mid-group: its last records are
        # later than the resume point stored in it. Scenes are written in time order, so the last record is the
        # true resume point; starting before it would compute those scenes twice.
        cursor = base["last"]
    window = base["window"] if base else None
    found = list_checkpoints(key)
    chain, merged, orphans = [], [], []
    for k, (index, path) in enumerate(found):
        st, why = read_state(path, DRIVE_READ_TRIES)
        if st is None:
            _stop_unusable(key, path, why)
        _same_settings(key, path, st)
        window = st["window"] if window is None else window
        end = st["resume_after"]
        if st["status"] != "checkpoint" or end is None or st["window"] != window:
            _stop_unusable(key, path, "is not a checkpoint of this path/row's window")
        if not chain and cursor is not None and end <= cursor:
            merged.append(path)                  # its records are already inside the path/row's file
            continue
        follows = st["previous"] == cursor and (not st["n"] or (st["last"] <= end and
                                                               (cursor is None or st["first"] > cursor)))
        if not follows and cursor is None:
            # It continues from an earlier point, but nothing on Drive ends there. Starting the path/row
            # again would throw that earlier work away, so this is never decided here.
            raise PipelineStop(
                f"[{key}] {os.path.relpath(path, DRIVE_OUT)} continues from {date_of(st['previous'])}, but nothing on "
                f"Drive ends there: {pr_filename(key)} or an earlier checkpoint is missing. Nothing was changed. It may "
                "still be uploading from another session, or be in Drive's trash - put it back and run again. If it "
                f"is gone for good, move the folder {os.path.relpath(ckpt_dir(key), DRIVE_OUT)} out of the output "
                "folder and run again: this path/row then starts from its first scene.")
        if not follows:                          # the checkpoint before this one never reached Drive
            orphans = [p for _, p in found[k:]]
            break
        chain.append(dict(st, index=index, path=path))
        cursor = end
    if orphans:
        first = os.path.basename(orphans[0])
        if len(orphans) > MAX_ORPHAN_CHECKPOINTS:
            raise PipelineStop(
                f"[{key}] {first} does not start where the checkpoint before it ended, so it and the checkpoints "
                f"after it ({len(orphans)} files) cannot be used as they are. Nothing was changed. If a checkpoint "
                f"was deleted, put it back. Otherwise move {first} and every later checkpoint out of "
                f"{os.path.relpath(ckpt_dir(key), DRIVE_OUT)} and run again: the run then continues after "
                f"{date_of(cursor)}.")
        aside = os.path.join(ckpt_dir(key), f"set_aside_{utcnow():%Y%m%dT%H%M%S}")
        os.makedirs(aside, exist_ok=True)
        for p in orphans:
            os.replace(p, os.path.join(aside, os.path.basename(p)))
        log.warning("[%s] %d checkpoint(s) from %s on do not follow the one before them (Drive had not finished "
                    "saving an earlier one when the last session ended). Moved to %s; their scenes are done again.",
                    key, len(orphans), first, os.path.relpath(aside, DRIVE_OUT))
    return dict(key=key, drive_nc=drive_nc, base=base, chain=chain, merged=merged, cursor=cursor, window=window,
                next_index=found[-1][0] + 1 if found else 1)


def remove_merged(key, merged, drive_nc):
    """Delete checkpoints whose records are inside the path/row's file, but only once that file has
    been on Drive for KEEP_CHECKPOINTS_MIN. Drive uploads in the background: if the runtime dies
    before the file is really up, the checkpoints must still be there to rebuild it from."""
    if not merged or (time.time() - os.path.getmtime(drive_nc)) < KEEP_CHECKPOINTS_MIN * 60:
        return 0
    for p in merged:
        os.remove(p)
    for name in os.listdir(ckpt_dir(key)):
        if name.endswith(".tmp"):
            os.remove(os.path.join(ckpt_dir(key), name))
    if not os.listdir(ckpt_dir(key)):
        os.rmdir(ckpt_dir(key))
    log.info("[%s] %d checkpoints removed from Drive (their records are in %s)", key, len(merged),
             os.path.basename(drive_nc))
    return len(merged)


def tidy_finished(pathrows):
    """Clean-up for path/rows finished earlier in this session (never allowed to disturb the run)."""
    for P in pathrows:
        t = FINISHED.get(P["key"])
        if t is None or time.time() - t < KEEP_CHECKPOINTS_MIN * 60:
            continue
        try:
            plan = resume_plan(P)
            remove_merged(P["key"], plan["merged"], plan["drive_nc"])
            FINISHED.pop(P["key"], None)
        except Exception as e:
            log.warning("[%s] checkpoint clean-up postponed: %s", P["key"], str(e)[:200])


# ---------- the end of a path/row: join its checkpoints into one file ----------
def assemble(P, W, base, chain, cursor, counts, t_start, work):
    """The path/row's file = the file already on Drive (if any) + every checkpoint, in order.
    One checkpoint at a time: fetch it from Drive, copy its records as stored, close the file,
    read them back from the closed file and compare, drop the scratch copy. Scratch therefore
    never holds more than the file being built and one checkpoint. The checkpoints stay on Drive."""
    key = P["key"]
    drive_nc, local_nc = os.path.join(DRIVE_OUT, pr_filename(key)), os.path.join(work, pr_filename(key))
    if not base and not sum(c["n"] for c in chain):
        log.warning("[%s] no scene with a usable pixel - no file written (its checkpoints keep the run from "
                    "downloading these scenes again)", key)
        return dict(key=key, status="no data", **counts)
    log.info("[%s] all scenes done - joining %d checkpoint file(s)%s into %s (window %d x %d cells, %d inside the aquifer)",
             key, len(chain), " and the file already on Drive" if base else "", pr_filename(key), W["shape"][0],
             W["shape"][1], W["n_roi"])
    total = (base["size"] if base else 0) + sum(c["size"] for c in chain)
    ensure_disk_free(f"{key} joining checkpoints",
                     extra_gb=(1.1 * total + max([c["size"] for c in chain], default=0)) / 1024 ** 3)
    t_j = time.time()
    if base:
        restore_from_drive(drive_nc, local_nc)
        st, why = read_state(local_nc)
        if st is None or (st["n"], st["resume_after"]) != (base["n"], base["resume_after"]):
            raise PipelineStop(f"[{key}] the copy of {os.path.basename(drive_nc)} fetched from Drive is not the file "
                               f"that was read a moment ago ({why or 'it has changed'}). Nothing was changed - run again.")
        header_bytes = None
    else:
        create_pr_nc(local_nc, P, W)
        header_bytes = os.path.getsize(local_nc)
    writer = PRWriter(local_nc, base["resume_after"] if base else None)
    n0, joined = writer.n, 0
    writer.close()
    for c in chain:
        part = os.path.join(work, os.path.basename(c["path"]))
        copy_from_drive(c["path"], part)
        writer.open()
        try:
            got = append_records(writer, part)
            writer.cursor = c["resume_after"]
            writer.commit(status="in_progress")
        finally:
            writer.close()
        if got != c["n"]:
            raise DataIntegrityError(f"[{key}] {os.path.basename(part)} does not hold the {c['n']} records it had on Drive")
        joined += verify_merge(local_nc, [part])     # these records, read back from the closed file, bit for bit
        os.remove(part)
    writer.open()
    writer.cursor = cursor
    writer.commit(status="in_progress")
    writer.close()
    good, problems, info = audit_nc(local_nc)
    if info.get("records") != n0 + joined:
        raise DataIntegrityError(f"[{key}] the joined file holds {info.get('records')} records, expected {n0 + joined}")
    if good:
        writer.open()
        writer.commit(status="complete")
        writer.close()
    else:
        log.error("[%s] AUDIT FAILED - uploaded as in_progress: %s", key, problems)
    TIMES["join checkpoints"] = TIMES.get("join checkpoints", 0.0) + time.time() - t_j
    save_to_drive(local_nc, drive_nc, key, os.path.basename(drive_nc))
    FINISHED[key] = time.time()
    size = os.path.getsize(local_nc)
    summary = dict(key=key, status="complete" if good else "audit failed", records=info.get("records", 0),
                   size=size, header_bytes=header_bytes, seconds=round(time.time() - t_start),
                   checkpoints_joined=len(chain), problems=problems, **counts)
    log_events(LOG_FILES, [summary])
    log.info("[%s] DONE: %d records (%d checkpoints joined and compared), %s on Drive, %d skipped, %d failed, %.0f min",
             key, summary["records"], len(chain), human(size), counts["skipped"], counts["failed"],
             (time.time() - t_start) / 60)
    return summary


def process_pathrow(P, prog):
    """One WRS-2 path/row from its resume point to the end of the record:
    download a group of scenes to scratch -> check -> index maths -> checkpoint on Drive -> delete -> repeat,
    then join the checkpoints into the path/row's file."""
    key, scenes = P["key"], P["scenes"]
    forced_skip, t_start = set(SKIP_SCENES), time.time()
    plan = resume_plan(P)
    base, cursor, chain = plan["base"], plan["cursor"], list(plan["chain"])
    have = (base["n"] if base else 0) + sum(c["n"] for c in chain)
    n_before = sum(1 for r in scenes if cursor is not None and scene_seconds(r) <= cursor)
    if n_before == len(scenes) and not chain and base and base["status"] == "complete":
        remove_merged(key, plan["merged"], plan["drive_nc"])
        prog.add(len(scenes), resumed=True)
        log.info("[%s] complete on Drive (%d records, through %s, %s) - skipped", key, base["n"],
                 date_of(base["last"]), human(base["size"]))
        return dict(key=key, status="already complete", records=base["n"], size=base["size"])
    work = os.path.join(NC_DIR, key)
    for d in (work, os.path.join(RAW_DIR, key)):
        shutil.rmtree(d, ignore_errors=True)     # scratch left by an earlier run is never used
    os.makedirs(work, exist_ok=True)
    W = window_ctx(key, plan["window"] or P["win"])  # the window is the one the files on Drive were made with
    prog.add(n_before, resumed=True)
    if base or chain:
        log.info("[%s] RESUME: Drive holds %d records through %s (%s) - %d of %d scenes still to do", key, have,
                 date_of(cursor), " + ".join(([f"{base['n']} in the path/row's file"] if base else []) +
                                             ([f"{have - (base['n'] if base else 0)} in {len(chain)} checkpoint file(s)"]
                                              if chain else [])), len(scenes) - n_before, len(scenes))
    else:
        log.info("[%s] nothing on Drive for this path/row yet: it starts at its first scene (%s). %s", key,
                 scenes[0]["t"][:10], "Each path/row has its own file and begins in the 1980s.")
    LAST.update(key=key, cursor=cursor, records=have)

    index, counts, maths_failures = plan["next_index"], dict(written=0, skipped=0, failed=0), {}
    checks_left = len(scenes) if TEST_MODE else READBACK_CHECKS
    waits, ck = 0, None
    try:
        while True:
            rest = [r for r in scenes if cursor is None or scene_seconds(r) > cursor]
            todo = rest[:GROUP_SCENES]
            while len(todo) < len(rest) and rest[len(todo)]["t"] == todo[-1]["t"]:
                todo.append(rest[len(todo)])     # scenes with one start time are never split over two groups
            if not todo:
                break
            fetch = [r for r in todo if r["id"] not in forced_skip]
            ok, permanent, transient = download_group(fetch, W) if fetch else ([], [], [])
            permanent += [(r, "listed in SKIP_SCENES") for r in todo if r["id"] in forced_skip]
            # Nothing at or after the first unfetched scene may be saved: the resume point only
            # ever moves over scenes that are written, skipped or proven unavailable.
            cut = min(scene_seconds(r) for r, _ in transient) if transient else None
            ok = sorted((x for x in ok if cut is None or scene_seconds(x[0]) < cut), key=lambda x: scene_seconds(x[0]))

            handled, skipped_log = [], []
            ck_path = os.path.join(work, f"checkpoint_{index:05d}.nc")
            ck = open_checkpoint(ck_path, P, W, index, cursor)
            if ok:
                t_m = time.time()
                jobs = [(r, parts, W) for r, parts in ok]
                # On a GPU the maths runs here, in the main thread, so that one thread owns the device for the
                # whole run. On the CPU, COMPUTE_THREADS scenes are computed ahead in worker threads.
                ex = ThreadPoolExecutor(COMPUTE_THREADS) if XP is np else None
                finished = False
                try:
                    for (r, parts), res in zip(ok, ex.map(_compute_safely, jobs) if ex else map(_compute_safely, jobs)):
                        if "error" in res:
                            maths_failures[r["id"]] = maths_failures.get(r["id"], 0) + 1
                            if maths_failures[r["id"]] >= 2:      # not a download problem: retrying cannot help
                                raise PipelineStop(f"The index maths failed twice on {r['id']}: {res['error']}. "
                                                   "Everything before this scene is saved. Please report this "
                                                   "message; to continue past the scene add its id to SKIP_SCENES.")
                            transient.append((r, "index maths failed: " + res["error"]))
                            cut = scene_seconds(r) if cut is None else min(cut, scene_seconds(r))
                            break
                        st = res["stats"]
                        TOTALS["px_clear"] = TOTALS.get("px_clear", 0) + st["clear"]
                        TOTALS["px_out_of_range"] = TOTALS.get("px_out_of_range", 0) + st["out_of_range"]
                        TOTALS["px_nonfinite"] = TOTALS.get("px_nonfinite", 0) + st["nonfinite"]
                        if st["cells_with_value"] == 0:
                            counts["skipped"] += 1
                            skipped_log.append(dict(scene=r["id"], pathrow=key, stage="no usable pixel inside the aquifer",
                                                    cloud_cover=r["cloud"], pixels_present=st["valid"]))
                        elif ck.append(r, res):
                            counts["written"] += 1
                            if checks_left > 0:
                                readback_check(ck, ck.n - 1, res, W)
                                checks_left -= 1
                        else:
                            counts["skipped"] += 1
                            skipped_log.append(dict(scene=r["id"], pathrow=key, stage="duplicate or out-of-order time"))
                        handled.append(r)
                    else:
                        finished = True
                finally:
                    if ex is not None:           # normal end: the workers are done; a stop does not wait for them
                        ex.shutdown(wait=finished, cancel_futures=not finished)
                TIMES["index maths + write"] = TIMES.get("index maths + write", 0.0) + time.time() - t_m
            permanent = [x for x in permanent if cut is None or scene_seconds(x[0]) < cut]
            handled += [x[0] for x in permanent]
            if handled:
                # logs first, checkpoint second: a run that dies in between repeats a log line, never loses one
                if permanent:
                    counts["failed"] += len(permanent)
                    log_events(LOG_FAILED, [dict(scene=r["id"], pathrow=key, reason=why) for r, why in permanent])
                    for r, why in permanent:
                        log.warning("[%s] permanently unavailable: %s (%s)", key, r["id"], why[:160])
                log_events(LOG_SKIPPED, skipped_log)
                reached = max(scene_seconds(r) for r in handled)
                ck.cursor = reached
                ck.commit()
                ck.close()
                dst = os.path.join(ckpt_dir(key), os.path.basename(ck_path))
                save_to_drive(ck_path, dst, key, f"checkpoint {index}")      # from here on this group is safe
                chain.append(dict(path=dst, index=index, n=ck.n, resume_after=reached, size=os.path.getsize(ck_path)))
                os.remove(ck_path)               # Drive has it; scratch keeps nothing
                TOTALS["records"] = TOTALS.get("records", 0) + ck.n
                TOTALS["checkpoints"] = TOTALS.get("checkpoints", 0) + 1
                have, cursor, index, ck = have + ck.n, reached, index + 1, None
                LAST.update(key=key, cursor=cursor, records=have)
                prog.add(len(handled))
                STALL.progress()
                waits = 0
                n_done = sum(1 for r in scenes if scene_seconds(r) <= cursor)
                so_far = (base["size"] if base else 0) + sum(c["size"] for c in chain)
                log.info("[%s] saved through %s | written %d, skipped %d, failed %d | %s | %s downloaded | free %.0f GB "
                         "| this path/row will take about %s on Drive", key, date_of(cursor), counts["written"],
                         counts["skipped"], counts["failed"], prog.line(), human(TOTALS.get("bytes", 0)), free_gb(),
                         human(so_far * len(scenes) / max(1, n_done)))
            else:
                ck.close()
                os.remove(ck_path)
                ck = None
            remove_raw(todo, key)                # scratch never holds more than one group of scenes
            if XP is not np:
                XP.get_default_memory_pool().free_all_blocks()
            gc.collect()
            if transient:
                r0_, why0 = min(transient, key=lambda x: scene_seconds(x[0]))
                log.warning("[%s] %d scene(s) could not be completed; first: %s (%s)", key, len(transient), r0_["id"], why0[:200])
                if STALL.failure():
                    raise PipelineStop(
                        f"No progress for {STALL_MINUTES} min; stuck at {r0_['id']} ({why0[:200]}). Everything so far "
                        "is saved - re-run to resume. If this same scene blocks every run while Earth Engine is "
                        "otherwise fine, add its id to SKIP_SCENES.")
                waits += 1
                time.sleep(min(30 * 2 ** waits, 600))

        return assemble(P, W, base, chain, cursor, counts, t_start, work)
    finally:
        # Nothing to rescue here: every finished group is already on Drive. A group that was cut
        # short is dropped and done again by the next run.
        try:
            if ck is not None:
                ck.close()
        except Exception:
            pass
        shutil.rmtree(os.path.join(RAW_DIR, key), ignore_errors=True)
        shutil.rmtree(work, ignore_errors=True)


# ---------- before any work: what is on Drive ----------
def drive_overview(pathrows):
    """One look at the output folder, so the log says plainly what this run will continue."""
    by_key = {P["key"]: P for P in pathrows}
    aside = {}
    for path in sorted(glob.glob(os.path.join(DRIVE_OUT, f"{PRODUCT_TAG}_p*_{ROI_LABEL}.nc.unreadable_*"))):
        m = _ASIDE_RE.search(os.path.basename(path))
        if m and m.group(1) + m.group(2) in by_key:
            aside.setdefault(m.group(1) + m.group(2), []).append(path)
    for key, paths in sorted(aside.items()):
        # The first version of this notebook renamed a path/row's file like this when Drive failed to open
        # it, and then started the path/row again. Such a file is normally intact.
        usable = []
        for p in paths:
            st, why = read_state(p, DRIVE_READ_TRIES)
            if st is None or st["signature"] != SETTINGS_SIGNATURE or not st["mask_ok"] or st["resume_after"] is None:
                log.warning("[%s] %s (set aside by an earlier version) cannot be used: %s - left where it is", key,
                            os.path.basename(p), why or "made with other settings")
            else:
                usable.append((st["resume_after"], p, st["n"]))
        if not usable:
            continue
        end, best, n = max(usable)
        target = os.path.join(DRIVE_OUT, pr_filename(key))
        if not os.path.exists(target) and not list_checkpoints(key):
            os.replace(best, target)
            log.warning("[%s] put back %s (%d records, through %s): an earlier version of this notebook had set it "
                        "aside after Drive failed to open it", key, os.path.basename(target), n, date_of(end))
            continue
        now = resume_plan(by_key[key])["cursor"]
        if now is None or now < end:
            raise PipelineStop(
                f"[{key}] {os.path.basename(best)} was set aside by an earlier version of this notebook and holds "
                f"more ({n} records, through {date_of(end)}) than what is in use now (through {date_of(now)}). "
                f"To continue from the larger one: in Drive delete {pr_filename(key)} and the folder "
                f"{CKPT_FOLDER}/p{key[:3]}r{key[3:]} if they exist, then run again; the set-aside file is put back "
                "by itself. To keep what is in use now, delete the set-aside file instead.")
        log.info("[%s] %s holds nothing that is not already on Drive under the proper name - it can be deleted",
                 key, os.path.basename(best))
    files = sum(os.path.exists(os.path.join(DRIVE_OUT, pr_filename(k))) for k in by_key)
    under_way = [(k, len(list_checkpoints(k))) for k in by_key]
    under_way = [f"{k} ({n} checkpoints)" for k, n in under_way if n]
    log.info("On Drive before this run: %d of %d path/rows have their file; checkpoints for %d path/row(s)%s",
             files, len(by_key), len(under_way), ": " + ", ".join(under_way[:6]) if under_way else "")


def preflight(pathrows):
    """Fail fast before any bulk work: Drive is writable, the disk floor holds, the engine is
    right at the chosen precision, and the local maths reproduces eemont for every satellite."""
    local, remote = os.path.join(NC_DIR, "_write_test.txt"), os.path.join(DRIVE_OUT, "_write_test.txt")
    with open(local, "w") as f:
        f.write(iso(utcnow()))
    robust_drive_copy(local, remote)
    os.remove(remote)
    os.remove(local)
    ensure_disk_free("preflight")
    engine_self_test()
    checks = []
    for sat in SATELLITES:
        cand = [r for P in pathrows for r in P["scenes"] if r["sat"] == sat and r["id"] not in SKIP_SCENES]
        if not cand:
            continue
        cand.sort(key=lambda r: (r["cloud"] < 0, r["cloud"]))
        for r in cand[:3]:                       # the clearest scenes of this satellite in this run
            try:
                res = eemont_cross_check(r)
            except SceneUnavailable:
                res = None
            if res:
                checks.append(res)
                break
    if not checks:
        raise PipelineStop("PREFLIGHT FAILED: no scene could be compared with eemont on Earth Engine.")
    log_events("eemont_cross_check.jsonl", checks)
    log.info("PREFLIGHT OK (%d satellite(s) checked against eemont)", len(checks))


def record_counts():
    """Scenes of the whole record over the aquifer's outline, per satellite (for the projection)."""
    region = ee.Geometry(json.loads(ROI_REGION))
    try:
        return {s: int(ee.ImageCollection(collection_id(s)).filterBounds(region).size().getInfo()) for s in SATELLITES}
    except Exception as e:
        log.warning("Could not count the whole record: %s", e)
        return None


def run_pipeline():
    t0 = time.time()
    TIMES.clear()
    TOTALS.clear()
    LAST.clear()
    log.info("=" * 78)
    log.info("  Landsat C2 L2 indices | %s | %d indices | TEST_MODE=%s", "+".join(SATELLITES), len(INDICES), TEST_MODE)
    log.info("  engine %s, pixel maths in %s | output: %s", ENGINE, np.dtype(CDTYPE).name, DRIVE_OUT)
    log.info("=" * 78)
    results, stopped, prog, listed = [], None, None, 0
    try:
        ee_login()
        load_scale_params()
        ensure_disk_free("start")
        pathrows = build_pathrows(get_manifest())
        listed = sum(len(P["scenes"]) for P in pathrows)
        log.info("%d path/rows hold aquifer cells, %d scenes", len(pathrows), listed)
        if ONLY_PATHROWS:
            wanted = set(ONLY_PATHROWS)
            absent = wanted - {P["key"] for P in pathrows}
            if absent:
                log.warning("ONLY_PATHROWS not available: %s", sorted(absent))
            pathrows = [P for P in pathrows if P["key"] in wanted]
        if TEST_MODE and pathrows:
            n_pick = min(TEST_PATHROWS, len(pathrows))
            pathrows = [pathrows[int((i + 0.5) * len(pathrows) / n_pick)] for i in range(n_pick)]
            for P in pathrows:
                pick = sorted({int(round(v)) for v in np.linspace(0, len(P["scenes"]) - 1, min(TEST_SCENES, len(P["scenes"])))})
                P["scenes"] = [P["scenes"][i] for i in pick]
        if not pathrows:
            raise PipelineStop("No path/row to process - check SATELLITES, the dates and ONLY_PATHROWS.")
        prog = Progress(sum(len(P["scenes"]) for P in pathrows))
        drive_overview(pathrows)
        preflight(pathrows)
        for k, P in enumerate(pathrows, 1):
            log.info("-" * 78)
            log.info("PATH/ROW %d/%d  %s  (%d aquifer cells in its window, %d scenes listed)", k, len(pathrows),
                     P["key"], P["n_roi"], len(P["scenes"]))
            results.append(process_pathrow(P, prog))
            tidy_finished(pathrows)
    except PipelineStop as e:
        stopped = str(e)
    except KeyboardInterrupt:
        stopped = "Stopped with the stop button."

    done = [r for r in results if r.get("status") == "complete"]
    written, size = sum(r.get("written", 0) for r in done), sum(r.get("size", 0) for r in done)
    log.info("=" * 78)
    if stopped is not None:
        if not LAST.get("key"):
            where = ""
        elif LAST["cursor"] is None:
            where = (f"\n  No group of path/row {LAST['key']} was finished, so nothing of it is on Drive yet; the next "
                     "run starts it at its first scene.")
        else:
            where = (f"\n  Path/row {LAST['key']} is saved on Drive through {date_of(LAST['cursor'])} "
                     f"({LAST['records']} records); the next run continues from there.")
        print("\n" + "=" * 78 + f"\n  STOPPED (nothing is lost, re-run to resume)\n  {stopped}{where}\n  Give Drive a "
              "minute to finish uploading before you disconnect the runtime.\n" + "=" * 78 + "\n")
    log.info("  path/rows: %s", {s: sum(1 for r in results if r.get("status") == s)
                                 for s in sorted({r.get("status") for r in results})})
    log.info("  this run: %d records saved in %d checkpoints, %d path/row file(s) finished (%s), %s downloaded in %d "
             "blocks (%d with reflectance), %.1f min", TOTALS.get("records", 0), TOTALS.get("checkpoints", 0), len(done),
             human(size), human(TOTALS.get("bytes", 0)), TOTALS.get("blocks", 0), TOTALS.get("sr_blocks", 0),
             (time.time() - t0) / 60)
    if TOTALS.get("px_clear"):
        log.info("  pixels averaged: %s | left out for the valid-range rule: %.3f %% | not a finite number: %d",
                 f"{TOTALS['px_clear']:,}", 100 * TOTALS.get("px_out_of_range", 0)
                 / (TOTALS["px_clear"] + TOTALS.get("px_out_of_range", 0)), TOTALS.get("px_nonfinite", 0))
    if TIMES:
        total_t = sum(TIMES.values())
        log.info("  time by stage: %s", ", ".join(f"{k} {v:.0f} s ({100 * v / total_t:.0f} %)"
                                                   for k, v in sorted(TIMES.items(), key=lambda kv: -kv[1])))
    fresh = [r for r in done if r.get("header_bytes")]
    if TEST_MODE and prog is not None and prog.run and fresh and sum(r["written"] for r in fresh):
        total = record_counts()
        if total:
            n_all = sum(total.values())
            w = sum(r["written"] for r in fresh)
            per_record = sum(r["size"] - r["header_bytes"] for r in fresh) / w
            per_scene = sum(TIMES.get(k, 0.0) for k in ("download", "index maths + write")) / prog.run
            log.info("  PROJECTION for the whole record (from this rehearsal, rough; summer scenes are clearer than average):")
            log.info("     %d scenes: %s", n_all, ", ".join(f"{SAT_NAME[s]} {n}" for s, n in total.items()))
            log.info("     ~%.0f h of runtime on this runtime type: %.1f s per scene (%s)", n_all * per_scene / 3600,
                     per_scene, ENGINE.split(" (")[0])
            log.info("     ~%.1f TB to download: %s per scene", n_all * TOTALS.get("bytes", 0) / prog.run / 1024 ** 4,
                     human(TOTALS.get("bytes", 0) / prog.run))
            log.info("     ~%.0f GB on Drive: %s per written record, %.0f %% of scenes written",
                     n_all * (written / prog.run) * per_record / 1024 ** 3, human(per_record), 100 * written / prog.run)
    log.info("=" * 78)
    return results


RESULTS = run_pipeline()

In [ ]:
# ═══════════════ CELL 10: audit every path/row file on Drive, and what is still under way ═══════════════
def _unique_scenes(name):
    """Scenes named in a log (a group that was done twice after a dead runtime is logged twice)."""
    path, seen = os.path.join(DRIVE_OUT, name), set()
    if os.path.exists(path):
        for line in open(path):
            try:
                seen.add(json.loads(line).get("scene"))
            except ValueError:
                pass
    return len(seen)


def audit_all():
    paths = sorted(glob.glob(os.path.join(DRIVE_OUT, f"{PRODUCT_TAG}_p*_{ROI_LABEL}.nc")))
    print(f"{len(paths)} path/row files in {DRIVE_OUT}\n")
    print(f"{'file':52s} {'status':12s} {'records':>8s} {'size':>10s}  first .. last scene")
    total, bad = 0, 0
    for p in paths:
        ok, problems, info = audit_nc(p)
        total += info.get("size", 0)
        print(f"{os.path.basename(p):52s} {info.get('status', '?'):12s} {info.get('records', 0):8d} "
              f"{human(info.get('size', 0)):>10s}  {info.get('first', '-')} .. {info.get('last', '-')}")
        if not ok:
            bad += 1
            for msg in problems:
                print(f"      PROBLEM: {msg}")
    print(f"\nTotal {human(total)} | {bad} file(s) with problems")

    # path/rows that are not finished: their records are in the checkpoint folder
    folders = sorted(glob.glob(os.path.join(DRIVE_OUT, CKPT_FOLDER, "p*r*")))
    rows = []
    for d in folders:
        key = os.path.basename(d)[1:4] + os.path.basename(d)[5:8]
        cks = list_checkpoints(key)
        if not cks:
            continue
        size = sum(os.path.getsize(p) for _, p in cks)
        last, why = read_state(cks[-1][1])
        final, _ = read_state(os.path.join(DRIVE_OUT, pr_filename(key))) if os.path.exists(
            os.path.join(DRIVE_OUT, pr_filename(key))) else (None, None)
        if last is None:
            note = f"last checkpoint unreadable: {why}"
        elif final and final["resume_after"] is not None and last["resume_after"] <= final["resume_after"]:
            note = "already inside the path/row's file; removed by a later run"
        else:
            note = f"under way, saved through {date_of(last['resume_after'])}"
        rows.append(f"{os.path.basename(d):10s} {len(cks):5d} checkpoints {human(size):>10s}  {note}")
    if rows:
        print(f"\nCheckpoint folders in {CKPT_FOLDER}/ (one small file per finished group of scenes):")
        print("\n".join("  " + r for r in rows))
    aside = sorted(glob.glob(os.path.join(DRIVE_OUT, "*.nc.unreadable_*")))
    if aside:
        print(f"\n{len(aside)} file(s) set aside by an earlier version of this notebook (see the start of the run log):")
        for p in aside:
            print("  " + os.path.basename(p))
    print()
    for name in (LOG_FAILED, LOG_SKIPPED):
        print(f"{name}: {_unique_scenes(name)} scenes")


audit_all()

# Reading a finished file: the index variables are ordinary floats, no decoding step is needed.
#   import xarray as xr
#   ds = xr.open_dataset("/content/drive/MyDrive/MSUGWB/LANDSAT_C2L2_indices_500m/<file>.nc")
#   ndvi = ds.NDVI.where(ds.n_clear >= 140)        # e.g. keep cells that are at least half clear
#
# Looking at a path/row that is still under way (its checkpoints, in order):
#   import glob, xarray as xr
#   parts = sorted(glob.glob("/content/drive/MyDrive/MSUGWB/LANDSAT_C2L2_indices_500m/_checkpoints/p031r031/checkpoint_*.nc"))
#   ds = xr.concat([xr.open_dataset(p) for p in parts], dim="time", data_vars="minimal", coords="minimal", compat="override")
#
# Optional independent CF check of one file:
# !pip -q install compliance-checker
# !compliance-checker --test=cf:1.9 "/content/drive/MyDrive/MSUGWB/LANDSAT_C2L2_indices_500m/<file>.nc"